# 예제 02. 반려와 재시도 — Orchestrator · Example 02 — Rejection and Retry: the Orchestrator

전산실험 · AI Agent 만들기 · **12주차** · 예상 소요 40분
Computational Experiments · Building AI Agents · **Week 12** · about 40 minutes

## 목표 · Objectives
- 세 역할을 엮어 돌린다
- 반려되면 되먹여 다시 시킨다
- **오류 누적**을 계산으로 확인한다
- 끝까지 반려되면 어떻게 할지 정한다

> **EN**
> - Wire the three roles together and run them
> - Feed a rejection back and try again
> - Confirm **how errors compound**, by calculation
> - Decide what to do when it is rejected to the very end

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

EN — Shared LLM wrapper for "Computational Experiments 2: Building AI Agents".
Calls the Keimyung Statistics department server and Google Gemini the same way.
Both expose an OpenAI-compatible API, so we use the single openai SDK and only
swap `base_url` and `model`.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기

 EN — This repository is PUBLIC, so never write the department server URL or
 the API key in code. Both are announced in class; store them in Colab Secrets.
     Key icon (🔑) on the left → new secret → turn on notebook access
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예 · Example:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
                                         # Check the server is alive first
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로 / Fallback route
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정 / Server configuration
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
# EN — The department server URL is never written here; it is read from the
#      Colab secret KMU_BASE_URL. The model tag is harmless to publish, so it
#      stays a constant.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
# EN — Fallback route. Get a free key from Google AI Studio. (public URL)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다 / read from Secrets at runtime
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기 / Loading secrets
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    EN — Look up a configuration value in a safe order:
    1) Colab Secrets (key icon on the left)  2) environment variable  3) prompt.
    The point is to keep the value out of the source. The moment a notebook or
    module is pushed to GitHub, anything written in the code is world-readable —
    the internal server URL just as much as the API key.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
             EN — True hides the typed value (getpass). Pass False for values
             like a URL where you need to see typos.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수 / environment variable
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력 / type it in
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다 / ask only once per runtime
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)

    EN — Fetch the API key. Alias of get_secret, kept under the week-1 name.
    """
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.

    EN — Return the server URL with any trailing '/' removed. The department
    URL is never stored in the repository, so it comes from Secrets. If you
    saved the full '/chat/completions' path, it is trimmed for you.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    # EN — The openai SDK appends /chat/completions itself, so strip it here in
    #      case the whole path was saved as the secret.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용 / reuse within the session
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트 / LLM client
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    EN — A thin client wrapping any OpenAI-compatible server.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
        EN — Which server to use. Defaults to the department server.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
        EN — Override the defaults. You normally leave these alone.
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        # EN — Never print the URL; screen shares and screenshots leak it easily.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 / Core calls -------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용.

        EN — Return the full response object, so you can inspect `usage`
        (token counts).
        """
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다.

        EN — Send the message list and return only the response text.
        """
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과.

        EN — Yield the response chunk by chunk, giving the typing effect.
        """
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 / Convenience methods ---------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다.

        EN — A one-shot question, for when building `messages` by hand is
        tedious.
        """
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.

        EN — Retry transient network/server errors, so that a lab with many
        calls (a 30-repetition experiment, say) does not die on one failure.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반 / any network or server error
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다 / back off progressively
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다.

        EN — Ask the server how many input tokens the messages use; the output
        is capped at 1 token to avoid waste.
        """
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수 / Module-level convenience functions
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.

    EN — The simplest one-line call. Internally it builds one LLM object and
    reuses it.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단 / Server diagnostics
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    EN — Check that the server is alive and see which models it serves. This is
    the very first lab of the course; on failure it prints what to check.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    # EN — Never print the full URL; show the host masked.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다.

    EN — Mask the host so screen shares and screenshots do not leak it.
    """
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들. / What to check when the connection fails."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile tools.py
"""전산실험2 5주차 : 도구(Tool)와 도구 레지스트리.

에이전트가 바깥 세계에 손을 뻗는 방법.

핵심 원칙:
    **계산은 모델이 아니라 도구가 한다.**

    LLM은 "어떤 도구를 어떤 인자로 부를지" 만 정한다.
    평균을 구하고 p값을 계산하는 것은 numpy 와 scipy 가 한다.
    1주차에 확인했듯 LLM은 난수도 제대로 못 뽑는다. 계산을 맡기면 안 된다.

구성:
    Tool          이름 · 설명 · 인자 명세 · 실제 함수
    ToolResult    실행 결과 (성공/실패, 값, 모델에게 돌려줄 관찰 문자열)
    ToolRegistry  도구를 등록·직렬화·호출한다
    StatToolbox   DataFrame 에 묶인 통계 도구 모음

EN — Week 5: tools and the tool registry — how an agent reaches into the world.
The governing principle is that THE TOOL COMPUTES, NOT THE MODEL. The LLM only
decides which tool to call with which arguments; numpy and scipy compute the
mean and the p-value. As week 1 showed, an LLM cannot even draw random numbers
properly, so never hand it the arithmetic.

The pieces: Tool (name, description, argument spec, the real function),
ToolResult (the outcome — success/failure, value, and the observation string
returned to the model), ToolRegistry (registers, serialises and calls tools),
and StatToolbox (statistics tools bound to one DataFrame).

사용 예 · Example:
    import pandas as pd
    from tools import StatToolbox

    box = StatToolbox(pd.read_csv("sample_clinic.csv"))
    print(box.describe())                     # 프롬프트에 넣을 도구 목록
    r = box.call("compare_two_groups",
                 {"value_column": "bmi", "group_column": "group"})
    print(r.observation())
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any, Callable

__all__ = ["Tool", "ToolResult", "ToolRegistry", "StatToolbox"]


# --------------------------------------------------------------------------
# 도구 정의 / Tool definition
# --------------------------------------------------------------------------
@dataclass
class Tool:
    """도구 하나. / A single tool.

    `parameters` 형식 · format::

        {
            "column": {"type": "str", "desc": "분석할 열 이름", "required": True},
            "alpha":  {"type": "float", "desc": "유의수준", "default": 0.05},
            "method": {"type": "str", "enum": ["pearson", "spearman"]},
        }
    """

    name: str
    description: str
    parameters: dict[str, dict[str, Any]]
    func: Callable[..., Any]

    def spec_line(self) -> str:
        """프롬프트에 넣을 한 줄 명세.

        EN — The one-line specification that goes into the prompt.
        """
        args = []
        for pname, p in self.parameters.items():
            t = p.get("type", "str")
            mark = "" if p.get("required") else "?"
            enum = f"={'|'.join(map(str, p['enum']))}" if "enum" in p else ""
            args.append(f"{pname}{mark}: {t}{enum}")
        return f"- {self.name}({', '.join(args)}) — {self.description}"

    def validate(self, args: dict[str, Any]) -> list[str]:
        """인자를 검사한다. 문제 목록을 돌려준다 (빈 목록 = 통과).

        3주차 `structured.validate()` 와 같은 발상이다.
        **모델이 준 것을 믿지 않고 코드가 확인한다.**

        EN — Check the arguments and return a list of problems; an empty list
        means they passed. Same idea as `structured.validate()` in week 3: the
        code verifies rather than trusting what the model produced.
        """
        problems: list[str] = []

        for pname, p in self.parameters.items():
            if p.get("required") and pname not in args:
                problems.append(f"필수 인자 '{pname}' 가 없습니다.")

        known = set(self.parameters)
        for given in args:
            if given not in known:
                problems.append(
                    f"'{given}' 는 이 도구의 인자가 아닙니다. 가능한 인자: {sorted(known)}"
                )

        for pname, value in args.items():
            p = self.parameters.get(pname)
            if p is None:
                continue
            if "enum" in p and value not in p["enum"]:
                problems.append(
                    f"'{pname}' 의 값 {value!r} 는 허용되지 않습니다. 가능한 값: {p['enum']}"
                )
            expected = p.get("type")
            if expected == "int" and not isinstance(value, int):
                problems.append(f"'{pname}' 는 정수여야 합니다 (받은 값: {value!r}).")
            elif expected == "float" and not isinstance(value, (int, float)):
                problems.append(f"'{pname}' 는 숫자여야 합니다 (받은 값: {value!r}).")
            elif expected == "bool" and not isinstance(value, bool):
                problems.append(f"'{pname}' 는 true/false 여야 합니다 (받은 값: {value!r}).")
            elif expected == "str" and not isinstance(value, str):
                problems.append(f"'{pname}' 는 문자열이어야 합니다 (받은 값: {value!r}).")
        return problems


@dataclass
class ToolResult:
    """도구 실행 결과. / The outcome of running a tool."""

    tool: str
    ok: bool
    value: Any = None
    error: str | None = None
    hint: str | None = None      # 실패했을 때 다음에 무엇을 하면 되는지 / what to try next after a failure

    def observation(self, max_chars: int = 1200) -> str:
        """모델에게 돌려줄 문자열.

        **실패도 관찰이다.** 조용히 넘기지 않고 무엇이 왜 안 됐는지 알려준다.
        그래야 모델이 다음 행동을 고칠 수 있다 (3주차 되먹임과 같은 구조).

        EN — The string handed back to the model. A FAILURE IS ALSO AN
        OBSERVATION: rather than swallowing it, we say what went wrong and why,
        so the model can correct its next action — the same feedback structure
        as week 3.
        """
        if not self.ok:
            msg = f"[{self.tool} 실패] {self.error}"
            if self.hint:
                msg += f"\n힌트: {self.hint}"
            return msg

        if isinstance(self.value, str):
            body = self.value
        else:
            body = json.dumps(self.value, ensure_ascii=False, indent=2, default=str)
        if len(body) > max_chars:
            body = body[:max_chars] + f"\n... (생략, 전체 {len(body)}자)"
        return f"[{self.tool} 결과]\n{body}"


# --------------------------------------------------------------------------
# 레지스트리 / Registry
# --------------------------------------------------------------------------
class ToolRegistry:
    """도구를 등록하고, 프롬프트용으로 직렬화하고, 호출한다.

    EN — Registers tools, serialises them for the prompt, and calls them.
    """

    def __init__(self) -> None:
        self._tools: dict[str, Tool] = {}
        self.log: list[ToolResult] = []      # 실행 이력 (6주차 trace 의 씨앗) / call history, the seed of the week-6 trace

    # -- 등록 / Registration ----------------------------------------------
    def register(self, tool: Tool) -> Tool:
        self._tools[tool.name] = tool
        return tool

    def tool(self, name: str, description: str, parameters: dict | None = None):
        """데코레이터로 등록한다.

            @registry.tool("mean", "평균을 구한다", {"col": {"type": "str", "required": True}})
            def _(col): ...

        EN — Register a tool with a decorator.
        """
        def deco(func: Callable[..., Any]) -> Callable[..., Any]:
            self.register(Tool(name, description, parameters or {}, func))
            return func

        return deco

    # -- 조회 / Lookup -----------------------------------------------------
    def names(self) -> list[str]:
        return sorted(self._tools)

    def __contains__(self, name: object) -> bool:
        return name in self._tools

    def __getitem__(self, name: str) -> Tool:
        return self._tools[name]

    def __len__(self) -> int:
        return len(self._tools)

    def describe(self) -> str:
        """프롬프트에 그대로 넣을 도구 목록.

        EN — The tool list, ready to paste straight into the prompt.
        """
        lines = [t.spec_line() for _, t in sorted(self._tools.items())]
        return "\n".join(lines)

    # -- 실행 / Execution --------------------------------------------------
    def call(self, name: str, args: dict[str, Any] | None = None) -> ToolResult:
        """도구를 실행한다. **예외를 밖으로 던지지 않는다.**

        에이전트 루프 안에서 도구가 예외를 던지면 루프 전체가 죽는다.
        실패를 `ToolResult` 로 감싸 돌려주면 모델이 보고 고칠 수 있다.

        EN — Run a tool. IT NEVER RAISES. If a tool raised inside the agent
        loop the whole loop would die; wrapping the failure in a `ToolResult`
        lets the model see it and fix it.
        """
        args = args or {}

        if name not in self._tools:
            result = ToolResult(
                name, False,
                error=f"'{name}' 라는 도구가 없습니다.",
                hint=f"사용 가능한 도구: {self.names()}",
            )
            self.log.append(result)
            return result

        tool = self._tools[name]
        problems = tool.validate(args)
        if problems:
            result = ToolResult(
                name, False,
                error="인자가 잘못되었습니다: " + "; ".join(problems),
                hint=tool.spec_line(),
            )
            self.log.append(result)
            return result

        try:
            value = tool.func(**args)
            result = ToolResult(name, True, value=value)
        except Exception as exc:
            result = ToolResult(
                name, False,
                error=f"{type(exc).__name__}: {exc}",
                hint="인자 값을 확인하세요. 열 이름은 list_columns 로 확인할 수 있습니다.",
            )
        self.log.append(result)
        return result


# --------------------------------------------------------------------------
# 통계 도구 모음 / Statistics toolbox
# --------------------------------------------------------------------------
class StatToolbox(ToolRegistry):
    """DataFrame 하나에 묶인 통계 도구들.

    도구는 **작고, 하나만 하고, 결과를 구조화해서** 돌려줘야 한다.
    거대한 만능 도구를 하나 만들면 모델이 인자를 제대로 채우지 못한다.

    EN — Statistics tools bound to a single DataFrame. A tool should be SMALL,
    do ONE thing, and return a STRUCTURED result. Build one giant do-everything
    tool and the model will fail to fill in its arguments.
    """

    def __init__(self, df: Any, name: str = "data") -> None:
        super().__init__()
        self.df = df
        self.name = name
        self._register_all()

    # -- 내부 헬퍼 / Internal helpers --------------------------------------
    def _numeric(self, column: str):
        """열을 수치로 강제 변환하고 결측을 뺀다.

        실제 데이터에는 '58세', ' 47 ' 같은 값이 섞여 있다.
        도구가 이것을 조용히 통과시키면 이후 계산이 전부 틀어진다.

        EN — Coerce a column to numeric and drop the missing values. Real data
        contains things like '58세' or ' 47 '; if a tool lets those slip through
        silently, every calculation downstream is wrong.
        """
        import pandas as pd

        if column not in self.df.columns:
            raise KeyError(f"'{column}' 열이 없습니다. 있는 열: {list(self.df.columns)}")
        s = pd.to_numeric(self.df[column], errors="coerce")
        n_bad = int(s.isna().sum() - self.df[column].isna().sum())
        s = s.dropna()
        if len(s) < 3:
            raise ValueError(f"'{column}' 의 유효한 수치가 {len(s)}개뿐입니다.")
        return s, n_bad

    def _check_columns(self, *columns: str) -> None:
        missing = [c for c in columns if c not in self.df.columns]
        if missing:
            raise KeyError(f"{missing} 열이 없습니다. 있는 열: {list(self.df.columns)}")

    # -- 도구 등록 / Registering the tools ---------------------------------
    def _register_all(self) -> None:
        import numpy as np
        import pandas as pd
        from scipy import stats

        # ---------------------------------------------------------------
        @self.tool(
            "list_columns",
            "데이터의 열 이름, 타입, 결측 수, 고유값 수를 돌려준다. 항상 먼저 호출할 것.",
            {},
        )
        def _list_columns() -> dict:
            rows = []
            for c in self.df.columns:
                s = self.df[c]
                num = pd.to_numeric(s, errors="coerce")
                rows.append({
                    "column": c,
                    "dtype": str(s.dtype),
                    "n_missing": int(s.isna().sum()),
                    "n_unique": int(s.nunique(dropna=True)),
                    "numeric_ok": bool(num.notna().sum() > len(s) * 0.8),
                    "sample": [str(v) for v in s.dropna().head(3).tolist()],
                })
            return {"n_rows": int(len(self.df)), "columns": rows}

        # ---------------------------------------------------------------
        @self.tool(
            "describe_column",
            "한 열의 기술통계를 돌려준다. 수치형이면 평균·표준편차·사분위수, 범주형이면 빈도표.",
            {"column": {"type": "str", "desc": "열 이름", "required": True}},
        )
        def _describe_column(column: str) -> dict:
            self._check_columns(column)
            s = self.df[column]
            num = pd.to_numeric(s, errors="coerce")

            if num.notna().sum() > len(s) * 0.8:      # 수치형으로 본다 / treat as numeric
                v = num.dropna()
                q1, q3 = v.quantile([0.25, 0.75])
                return {
                    "column": column, "kind": "numeric", "n": int(len(v)),
                    "n_missing": int(s.isna().sum()),
                    "n_non_numeric": int(num.isna().sum() - s.isna().sum()),
                    "mean": round(float(v.mean()), 4),
                    "sd": round(float(v.std(ddof=1)), 4),
                    "min": round(float(v.min()), 4),
                    "q1": round(float(q1), 4),
                    "median": round(float(v.median()), 4),
                    "q3": round(float(q3), 4),
                    "max": round(float(v.max()), 4),
                    "skew": round(float(v.skew()), 3),
                }
            counts = s.value_counts(dropna=True)
            return {
                "column": column, "kind": "categorical",
                "n": int(s.notna().sum()), "n_missing": int(s.isna().sum()),
                "n_levels": int(counts.size),
                "counts": {str(k): int(v) for k, v in counts.head(15).items()},
            }

        # ---------------------------------------------------------------
        @self.tool(
            "check_normality",
            "한 열의 정규성을 Shapiro-Wilk 로 검정한다. 검정 방법을 고르기 전에 확인할 것.",
            {"column": {"type": "str", "desc": "수치형 열 이름", "required": True},
             "group_column": {"type": "str", "desc": "집단별로 나눠 검정할 열 (선택)"}},
        )
        def _check_normality(column: str, group_column: str | None = None) -> dict:
            if group_column is None:
                v, n_bad = self._numeric(column)
                W, p = stats.shapiro(v[:5000])
                return {"column": column, "n": int(len(v)),
                        "n_non_numeric": n_bad,
                        "statistic_W": round(float(W), 4), "p_value": float(p),
                        "normal_at_0.05": bool(p >= 0.05)}

            self._check_columns(column, group_column)
            out = {}
            work = self.df[[column, group_column]].copy()
            work[column] = pd.to_numeric(work[column], errors="coerce")
            for g, sub in work.dropna().groupby(group_column):
                if len(sub) < 3:
                    out[str(g)] = {"n": int(len(sub)), "error": "표본이 3 미만"}
                    continue
                W, p = stats.shapiro(sub[column][:5000])
                out[str(g)] = {"n": int(len(sub)),
                               "statistic_W": round(float(W), 4),
                               "p_value": float(p),
                               "normal_at_0.05": bool(p >= 0.05)}
            return {"column": column, "by": group_column, "groups": out}

        # ---------------------------------------------------------------
        @self.tool(
            "compare_two_groups",
            "두 독립 집단의 평균(또는 분포)을 비교한다. 등분산은 Welch 로 보정하며, "
            "method=auto 면 정규성 검정 결과에 따라 t검정/Mann-Whitney 를 고른다.",
            {"value_column": {"type": "str", "desc": "비교할 수치형 열", "required": True},
             "group_column": {"type": "str", "desc": "두 수준을 가진 집단 열", "required": True},
             "method": {"type": "str", "enum": ["auto", "t", "mannwhitney"],
                        "desc": "기본 auto"},
             "alpha": {"type": "float", "desc": "유의수준, 기본 0.05"}},
        )
        def _compare_two_groups(value_column: str, group_column: str,
                                method: str = "auto", alpha: float = 0.05) -> dict:
            self._check_columns(value_column, group_column)
            work = self.df[[value_column, group_column]].copy()
            work[value_column] = pd.to_numeric(work[value_column], errors="coerce")
            work = work.dropna()

            levels = sorted(work[group_column].astype(str).unique())
            if len(levels) != 2:
                raise ValueError(
                    f"'{group_column}' 의 수준이 {len(levels)}개입니다 ({levels[:5]}). "
                    f"두 집단 비교에는 2개여야 합니다. 3개 이상이면 compare_many_groups 를 쓰세요."
                )
            a = work[work[group_column].astype(str) == levels[0]][value_column]
            b = work[work[group_column].astype(str) == levels[1]][value_column]

            chosen = method
            if method == "auto":
                pa = stats.shapiro(a[:5000]).pvalue if len(a) >= 3 else 0.0
                pb = stats.shapiro(b[:5000]).pvalue if len(b) >= 3 else 0.0
                chosen = "t" if (pa >= 0.05 and pb >= 0.05) else "mannwhitney"

            if chosen == "t":
                res = stats.ttest_ind(a, b, equal_var=False)      # Welch
                test_name = "Welch independent t-test"
                # Hedges 보정 없는 Cohen's d (pooled sd)
                # EN — Cohen's d with pooled sd, no Hedges correction
                sp = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1))
                             / (len(a) + len(b) - 2))
                effect = {"name": "Cohen's d", "value": round(float((a.mean() - b.mean()) / sp), 4)}
            else:
                res = stats.mannwhitneyu(a, b, alternative="two-sided")
                test_name = "Mann-Whitney U"
                # rank-biserial correlation
                r = 1 - (2 * res.statistic) / (len(a) * len(b))
                effect = {"name": "rank-biserial r", "value": round(float(r), 4)}

            return {
                "test": test_name,
                "method_requested": method, "method_used": chosen,
                "groups": {levels[0]: {"n": int(len(a)), "mean": round(float(a.mean()), 4),
                                       "sd": round(float(a.std(ddof=1)), 4),
                                       "median": round(float(a.median()), 4)},
                           levels[1]: {"n": int(len(b)), "mean": round(float(b.mean()), 4),
                                       "sd": round(float(b.std(ddof=1)), 4),
                                       "median": round(float(b.median()), 4)}},
                "statistic": round(float(res.statistic), 4),
                "p_value": float(res.pvalue),
                "alpha": alpha,
                "significant": bool(res.pvalue < alpha),
                "effect_size": effect,
            }

        # ---------------------------------------------------------------
        @self.tool(
            "compare_paired",
            "같은 대상에서 두 번 측정한 값(전/후)을 비교한다. 대응표본 t검정 또는 Wilcoxon.",
            {"before_column": {"type": "str", "desc": "이전 측정값 열", "required": True},
             "after_column": {"type": "str", "desc": "이후 측정값 열", "required": True},
             "method": {"type": "str", "enum": ["auto", "t", "wilcoxon"]},
             "alpha": {"type": "float"}},
        )
        def _compare_paired(before_column: str, after_column: str,
                            method: str = "auto", alpha: float = 0.05) -> dict:
            self._check_columns(before_column, after_column)
            work = self.df[[before_column, after_column]].apply(
                lambda c: pd.to_numeric(c, errors="coerce")).dropna()
            if len(work) < 3:
                raise ValueError(f"두 열이 모두 있는 행이 {len(work)}개뿐입니다.")
            before, after = work[before_column], work[after_column]
            diff = after - before

            chosen = method
            if method == "auto":
                chosen = "t" if stats.shapiro(diff[:5000]).pvalue >= 0.05 else "wilcoxon"

            if chosen == "t":
                res = stats.ttest_rel(before, after)
                test_name = "paired t-test"
                effect = {"name": "Cohen's dz",
                          "value": round(float(diff.mean() / diff.std(ddof=1)), 4)}
            else:
                res = stats.wilcoxon(before, after)
                test_name = "Wilcoxon signed-rank"
                effect = {"name": "median difference", "value": round(float(diff.median()), 4)}

            return {
                "test": test_name, "method_used": chosen,
                "n_pairs": int(len(work)),
                "mean_before": round(float(before.mean()), 4),
                "mean_after": round(float(after.mean()), 4),
                "mean_difference": round(float(diff.mean()), 4),
                "statistic": round(float(res.statistic), 4),
                "p_value": float(res.pvalue),
                "alpha": alpha, "significant": bool(res.pvalue < alpha),
                "effect_size": effect,
            }

        # ---------------------------------------------------------------
        @self.tool(
            "compare_many_groups",
            "세 집단 이상의 평균(또는 분포)을 비교한다. 일원분산분석 또는 Kruskal-Wallis.",
            {"value_column": {"type": "str", "required": True},
             "group_column": {"type": "str", "required": True},
             "method": {"type": "str", "enum": ["auto", "anova", "kruskal"]},
             "alpha": {"type": "float"}},
        )
        def _compare_many_groups(value_column: str, group_column: str,
                                 method: str = "auto", alpha: float = 0.05) -> dict:
            self._check_columns(value_column, group_column)
            work = self.df[[value_column, group_column]].copy()
            work[value_column] = pd.to_numeric(work[value_column], errors="coerce")
            work = work.dropna()

            groups = [g[value_column].values for _, g in work.groupby(group_column)]
            labels = [str(k) for k, _ in work.groupby(group_column)]
            if len(groups) < 3:
                raise ValueError(
                    f"집단이 {len(groups)}개입니다. 3개 이상이어야 합니다. "
                    f"2개면 compare_two_groups 를 쓰세요."
                )

            chosen = method
            if method == "auto":
                정규 = all(len(g) >= 3 and stats.shapiro(g[:5000]).pvalue >= 0.05 for g in groups)   # normality
                등분산 = stats.levene(*groups).pvalue >= 0.05                                        # equal variance
                chosen = "anova" if (정규 and 등분산) else "kruskal"

            if chosen == "anova":
                res = stats.f_oneway(*groups)
                test_name = "one-way ANOVA"
                grand = np.concatenate(groups)
                ss_b = sum(len(g) * (g.mean() - grand.mean()) ** 2 for g in groups)
                ss_t = ((grand - grand.mean()) ** 2).sum()
                effect = {"name": "eta squared", "value": round(float(ss_b / ss_t), 4)}
            else:
                res = stats.kruskal(*groups)
                test_name = "Kruskal-Wallis"
                n = len(np.concatenate(groups))
                effect = {"name": "epsilon squared",
                          "value": round(float(res.statistic / ((n ** 2 - 1) / (n + 1))), 4)}

            return {
                "test": test_name, "method_used": chosen,
                "groups": {lab: {"n": int(len(g)), "mean": round(float(g.mean()), 4),
                                 "sd": round(float(g.std(ddof=1)), 4)}
                           for lab, g in zip(labels, groups)},
                "statistic": round(float(res.statistic), 4),
                "p_value": float(res.pvalue),
                "alpha": alpha, "significant": bool(res.pvalue < alpha),
                "effect_size": effect,
                "note": "유의하면 사후검정(Tukey 등)이 필요합니다.",
            }

        # ---------------------------------------------------------------
        @self.tool(
            "test_association",
            "두 범주형 변수의 연관성을 카이제곱 검정한다. 기대빈도가 5 미만이면 Fisher 를 권한다.",
            {"column_a": {"type": "str", "required": True},
             "column_b": {"type": "str", "required": True},
             "alpha": {"type": "float"}},
        )
        def _test_association(column_a: str, column_b: str, alpha: float = 0.05) -> dict:
            self._check_columns(column_a, column_b)
            table = pd.crosstab(self.df[column_a], self.df[column_b])
            if table.size < 4:
                raise ValueError("분할표가 2x2 보다 작습니다.")
            chi2, p, dof, expected = stats.chi2_contingency(table)
            min_exp = float(expected.min())
            n = int(table.values.sum())
            cramers_v = float(np.sqrt(chi2 / (n * (min(table.shape) - 1))))
            return {
                "test": "chi-square test of independence",
                "table": table.to_dict(),
                "chi2": round(float(chi2), 4), "df": int(dof),
                "p_value": float(p), "alpha": alpha,
                "significant": bool(p < alpha),
                "min_expected_count": round(min_exp, 2),
                "effect_size": {"name": "Cramer's V", "value": round(cramers_v, 4)},
                "warning": (None if min_exp >= 5 else
                            "기대빈도가 5 미만인 칸이 있습니다. Fisher 정확검정을 고려하세요."),
            }

        # ---------------------------------------------------------------
        @self.tool(
            "correlate",
            "두 수치형 변수의 상관을 구한다. method=auto 면 정규성에 따라 pearson/spearman 선택.",
            {"column_a": {"type": "str", "required": True},
             "column_b": {"type": "str", "required": True},
             "method": {"type": "str", "enum": ["auto", "pearson", "spearman"]}},
        )
        def _correlate(column_a: str, column_b: str, method: str = "auto") -> dict:
            self._check_columns(column_a, column_b)
            work = self.df[[column_a, column_b]].apply(
                lambda c: pd.to_numeric(c, errors="coerce")).dropna()
            if len(work) < 3:
                raise ValueError(f"두 열이 모두 있는 행이 {len(work)}개뿐입니다.")
            a, b = work[column_a], work[column_b]

            chosen = method
            if method == "auto":
                정규 = (stats.shapiro(a[:5000]).pvalue >= 0.05
                        and stats.shapiro(b[:5000]).pvalue >= 0.05)
                chosen = "pearson" if 정규 else "spearman"

            fn = stats.pearsonr if chosen == "pearson" else stats.spearmanr
            res = fn(a, b)
            return {
                "method_used": chosen, "n": int(len(work)),
                "coefficient": round(float(res[0]), 4),
                "p_value": float(res[1]),
                "note": "상관은 인과가 아닙니다.",
            }


In [ ]:
%%writefile structured.py
"""전산실험2 3주차 : 구조화 출력 — 파싱 · 검증 · 재시도.

소형 모델은 "JSON으로만 답하라"고 지시해도 자주 어긴다.
코드펜스로 감싸거나, 인사를 덧붙이거나, 따옴표를 빠뜨린다.

이 모듈은 세 겹의 방어를 제공한다.

    1) 프롬프트로 형식을 강제하고        (build_json_prompt)
    2) 지저분한 응답에서 JSON을 긁어내고  (extract_json)
    3) 스키마로 검증한 뒤, 실패하면       (validate)
       오류를 되먹여 다시 요구한다        (ask_json)

EN — Week 3: structured output — parse, validate, retry.
Small models break the "answer in JSON only" rule all the time: they wrap the
answer in a code fence, add a greeting, or drop a quote. This module gives you
three layers of defence: force the format in the prompt (build_json_prompt),
scrape the JSON out of a messy reply (extract_json), validate it against a
schema (validate) and, on failure, feed the error back and ask again (ask_json).

사용 예 · Example:
    from kmu_llm import LLM
    from structured import ask_json

    schema = {
        "required": ["test", "reason"],
        "types": {"test": str, "reason": str},
        "allowed": {"test": ["t-test", "anova", "chi-square"]},
    }
    result = ask_json("두 그룹의 평균 비교에 쓸 검정은?", schema=schema)
    print(result["test"])
"""

from __future__ import annotations

import json
import re
from typing import Any

__all__ = [
    "JSONParseError",
    "SchemaError",
    "extract_json",
    "parse_json",
    "validate",
    "build_json_prompt",
    "ask_json",
]


class JSONParseError(ValueError):
    """모델 응답에서 JSON을 뽑아내지 못했을 때.

    EN — Raised when no JSON could be extracted from the model's reply.
    """


class SchemaError(ValueError):
    """JSON은 얻었지만 요구한 구조가 아닐 때.

    EN — Raised when JSON was obtained but does not match the required shape.
    """


# --------------------------------------------------------------------------
# 1) 지저분한 응답에서 JSON 긁어내기 / Scraping JSON out of a messy reply
# --------------------------------------------------------------------------
_FENCE = re.compile(r"```(?:json|python)?\s*(.*?)```", re.DOTALL)


def extract_json(text: str) -> str:
    """모델 응답에서 JSON 문자열만 뽑아낸다.

    소형 모델이 실제로 내놓는 형태들:
        ```json\\n{...}\\n```
        네, 알겠습니다! {...}
        {...}\\n도움이 되었길 바랍니다.

    EN — Pull just the JSON string out of the model's reply. The shapes above
    are what small models actually return: a fenced block, a greeting before
    the object, or a closing pleasantry after it.

    Returns
    -------
    str : JSON 으로 보이는 부분 / the part that looks like JSON

    Raises
    ------
    JSONParseError : 중괄호/대괄호 쌍을 찾지 못한 경우
                     EN — no matching brace/bracket pair was found
    """
    if not text or not text.strip():
        raise JSONParseError("응답이 비어 있습니다.")

    # (1) 코드펜스가 있으면 그 안쪽을 우선 사용
    #     EN — if there is a code fence, prefer what is inside it
    fence = _FENCE.search(text)
    if fence:
        text = fence.group(1)

    # (2) 가장 바깥쪽 { } 또는 [ ] 를 찾는다
    #     EN — find the outermost { } or [ ] pair
    for opener, closer in (("{", "}"), ("[", "]")):
        start = text.find(opener)
        end = text.rfind(closer)
        if start != -1 and end > start:
            return text[start : end + 1].strip()

    raise JSONParseError(
        f"JSON 구조를 찾지 못했습니다. 응답 앞부분: {text[:100]!r}"
    )


def parse_json(text: str) -> Any:
    """extract_json + json.loads. 실패하면 JSONParseError.

    EN — extract_json followed by json.loads; raises JSONParseError on failure.
    """
    snippet = extract_json(text)
    try:
        return json.loads(snippet)
    except json.JSONDecodeError as exc:
        # 흔한 사고: 마지막 항목 뒤의 쉼표(trailing comma)
        # EN — the usual accident: a trailing comma after the last item
        repaired = re.sub(r",\s*([}\]])", r"\1", snippet)
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            pass
        raise JSONParseError(
            f"JSON 파싱 실패 ({exc.msg}, {exc.lineno}행 {exc.colno}열). "
            f"문제 부분: {snippet[max(0, exc.pos - 30) : exc.pos + 30]!r}"
        ) from exc


# --------------------------------------------------------------------------
# 2) 스키마 검증 / Schema validation
# --------------------------------------------------------------------------
def validate(data: Any, schema: dict | None) -> list[str]:
    """아주 단순한 스키마 검증기. 문제점 목록을 돌려준다 (빈 목록 = 통과).

    스키마 형식::

        {
            "required": ["test", "reason"],          # 반드시 있어야 할 키
            "types":    {"alpha": float, "n": int},  # 값의 파이썬 타입
            "allowed":  {"test": ["t-test", ...]},   # 허용되는 값 목록
            "range":    {"alpha": (0.0, 1.0)},       # 수치 범위 (닫힌 구간)
        }

    pydantic 같은 라이브러리를 쓸 수도 있지만, 여기서는 검증이 무엇을 하는
    일인지 보이도록 직접 구현한다.

    EN — A deliberately tiny schema validator. It returns a list of problems;
    an empty list means the data passed. The schema keys above are: required
    keys, expected Python types, allowed value lists, and closed numeric
    ranges. A library such as pydantic would do this for us, but we write it by
    hand so you can see what validation actually is.
    """
    if schema is None:
        return []

    problems: list[str] = []

    if not isinstance(data, dict):
        return [f"최상위가 객체(dict)가 아니라 {type(data).__name__} 입니다."]

    for key in schema.get("required", []):
        if key not in data:
            problems.append(f"필수 키 '{key}' 가 없습니다.")

    for key, expected in schema.get("types", {}).items():
        if key not in data:
            continue
        value = data[key]
        # int 는 float 자리에 들어와도 허용한다 / an int is acceptable where a float is expected
        if expected is float and isinstance(value, int) and not isinstance(value, bool):
            continue
        if not isinstance(value, expected):
            problems.append(
                f"'{key}' 는 {expected.__name__} 여야 하는데 "
                f"{type(value).__name__} 입니다 (값: {value!r})."
            )

    for key, options in schema.get("allowed", {}).items():
        if key in data and data[key] not in options:
            problems.append(
                f"'{key}' 의 값 {data[key]!r} 는 허용 목록에 없습니다. "
                f"가능한 값: {options}"
            )

    for key, (low, high) in schema.get("range", {}).items():
        if key not in data:
            continue
        value = data[key]
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            problems.append(f"'{key}' 가 숫자가 아니어서 범위를 확인할 수 없습니다.")
        elif not (low <= value <= high):
            problems.append(f"'{key}' 의 값 {value} 가 [{low}, {high}] 범위 밖입니다.")

    return problems


# --------------------------------------------------------------------------
# 3) 프롬프트로 형식 강제하기 / Forcing the format through the prompt
# --------------------------------------------------------------------------
def build_json_prompt(prompt: str, schema: dict | None = None, example: Any = None) -> str:
    """JSON 출력을 유도하는 지시문을 프롬프트에 덧붙인다.

    소형 모델에서 실제로 효과가 있었던 순서대로 넣는다.
    1) 출력 예시  2) 금지 사항 명시  3) 시작·끝 문자 지정

    EN — Append instructions that steer the model toward JSON output, in the
    order that actually worked on small models: (1) show an example, (2) state
    what is forbidden, (3) pin down the first and last character.
    """
    parts = [prompt, "", "## 출력 형식"]

    if example is not None:
        parts.append("아래와 정확히 같은 형식의 JSON으로만 답하세요.")
        parts.append(json.dumps(example, ensure_ascii=False, indent=2))
    elif schema and schema.get("required"):
        keys = ", ".join(f'"{k}"' for k in schema["required"])
        parts.append(f"다음 키를 가진 JSON 객체로만 답하세요: {keys}")

    if schema:
        for key, options in schema.get("allowed", {}).items():
            parts.append(f'- "{key}" 는 다음 중 하나여야 합니다: {options}')

    parts += [
        "",
        "규칙:",
        "- 설명, 인사, 사족을 쓰지 마세요.",
        "- 코드블록 표시(백틱)를 쓰지 마세요.",
        "- 응답은 반드시 { 로 시작해서 } 로 끝나야 합니다.",
    ]
    return "\n".join(parts)


# --------------------------------------------------------------------------
# 4) 파싱 · 검증 · 재시도를 합친 것 / Parse + validate + retry, combined
# --------------------------------------------------------------------------
def ask_json(
    prompt: str,
    system: str | None = None,
    *,
    schema: dict | None = None,
    example: Any = None,
    n_retry: int = 3,
    llm: Any = None,
    verbose: bool = False,
    **kw: Any,
) -> Any:
    """모델에게 JSON을 받아 파싱·검증한 결과를 돌려준다.

    실패하면 **오류 메시지를 모델에게 되돌려주고 다시 요구한다.**
    이 되먹임 구조가 6주차 ReAct 루프의 핵심 아이디어와 같다.

    EN — Get JSON from the model and return the parsed, validated result. On
    failure the error message is fed back to the model and the request is
    repeated. That feedback loop is the same core idea as the week-6 ReAct loop.

    Parameters
    ----------
    schema  : validate() 형식의 스키마 / a schema in validate()'s format
    example : 출력 예시 (few-shot). 소형 모델에서 가장 효과가 크다
              EN — an output example (few-shot); the biggest win on small models
    n_retry : 최대 시도 횟수 / maximum number of attempts
    llm     : LLM 인스턴스. 없으면 kmu_llm.LLM() 을 만든다
              EN — an LLM instance; one is created if omitted
    verbose : 시도 과정을 출력 / print each attempt

    Raises
    ------
    JSONParseError / SchemaError : n_retry 회 모두 실패한 경우
                                   EN — when all n_retry attempts failed
    """
    if llm is None:
        from kmu_llm import LLM

        llm = LLM()

    kw.setdefault("temperature", 0.0)  # 형식을 지켜야 하므로 낮게 / low, because format matters

    messages: list[dict[str, str]] = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": build_json_prompt(prompt, schema, example)})

    last_error: Exception | None = None

    for attempt in range(1, n_retry + 1):
        raw = llm.chat(messages, **kw)
        if verbose:
            print(f"[시도 {attempt}] 원문: {raw[:120]!r}")

        try:
            data = parse_json(raw)
        except JSONParseError as exc:
            last_error = exc
            feedback = (
                f"방금 응답을 JSON으로 읽을 수 없었습니다. 오류: {exc}\n"
                "설명 없이 JSON 객체만 다시 출력하세요."
            )
        else:
            problems = validate(data, schema)
            if not problems:
                if verbose:
                    print(f"[시도 {attempt}] 성공")
                return data
            last_error = SchemaError("; ".join(problems))
            feedback = (
                "JSON 형식은 맞지만 내용에 문제가 있습니다:\n"
                + "\n".join(f"- {p}" for p in problems)
                + "\n이 문제들을 고쳐서 JSON 객체만 다시 출력하세요."
            )

        if verbose:
            print(f"[시도 {attempt}] 실패 -> 되먹임: {last_error}")

        # 실패한 응답과 오류를 대화에 남겨서 모델이 스스로 고치게 한다
        # EN — keep the failed reply and the error in the conversation so the
        #      model can correct itself
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": feedback})

    raise type(last_error)(  # type: ignore[misc]
        f"{n_retry}회 시도 모두 실패했습니다. 마지막 오류: {last_error}"
    )


In [ ]:
%%writefile react.py
"""전산실험2 6주차 : ReAct 루프 — 에이전트의 심장.

    Thought  → 지금 무엇을 알아야 하는가
    Action   → 어떤 도구를 어떤 인자로 부를 것인가
    Observation → 도구가 돌려준 결과 (실패도 관찰이다)
        ↓ 목표가 달성될 때까지 반복

이 구조는 3주차 `ask_json()` 의 되먹임 루프와 같다.
달라진 것은 "관찰"이 파싱 결과가 아니라 **도구 실행 결과**라는 점뿐이다.

설계에서 중요한 것 세 가지:
    1. 종료 조건을 여러 겹으로 둔다 (정답 / 스텝 상한 / 연속 실패 / 반복 감지)
    2. 실패를 루프 밖으로 던지지 않는다
    3. 모든 단계를 기록한다 (Trace) — 11주차 평가의 재료가 된다

EN — Week 6: the ReAct loop, the heart of an agent.
    Thought      → what do I need to know right now
    Action       → which tool do I call, with which arguments
    Observation  → what the tool returned (a failure is an observation too)
        ↓ repeat until the goal is met
This is the same shape as the feedback loop in week 3's `ask_json()`; the only
change is that the "observation" is now a tool result rather than a parse
result. Three things matter in the design: (1) layer several stop conditions
(answer found / step limit / consecutive failures / repetition detected),
(2) never let a failure escape the loop, and (3) record every step in a Trace —
that becomes the raw material for the week-11 evaluation.

사용 예 · Example:
    import pandas as pd
    from kmu_llm import LLM
    from tools import StatToolbox
    from react import ReActAgent

    box = StatToolbox(pd.read_csv("sample_clinic.csv"))
    agent = ReActAgent(LLM(), box, max_steps=8)
    trace = agent.run("치료 전후로 혈압이 유의하게 떨어졌나요?")
    print(trace.final_answer)
    trace.show()
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any

__all__ = ["Step", "Trace", "ReActAgent", "REACT_SYSTEM"]


# --------------------------------------------------------------------------
# 실행 기록 / Execution record
# --------------------------------------------------------------------------
@dataclass
class Step:
    """한 사이클의 기록. / The record of one cycle."""

    n: int
    thought: str = ""
    action: str | None = None
    args: dict[str, Any] = field(default_factory=dict)
    observation: str = ""
    ok: bool = False
    raw: str = ""           # 모델의 원문 (디버깅용) / the model's raw reply, for debugging
    error: str | None = None

    def summary(self) -> str:
        head = f"[{self.n}] {self.action}({json.dumps(self.args, ensure_ascii=False)})"
        mark = "OK" if self.ok else "실패"
        return f"{head}  -> {mark}"


@dataclass
class Trace:
    """한 번의 `run()` 전체 기록.

    11주차에서 이 객체를 모아 성능을 추정한다.

    EN — The full record of a single `run()`. In week 11 we collect these
    objects to estimate the agent's performance.
    """

    goal: str
    steps: list[Step] = field(default_factory=list)
    final_answer: str | None = None
    stop_reason: str = ""
    success: bool = False

    def __len__(self) -> int:
        return len(self.steps)

    @property
    def tools_used(self) -> list[str]:
        return [s.action for s in self.steps if s.action]

    @property
    def n_failed(self) -> int:
        return sum(1 for s in self.steps if not s.ok)

    def show(self, max_obs: int = 220) -> None:
        """사람이 읽는 실행 경로. / The execution path, for a human to read."""
        print(f"목표: {self.goal}")
        print("=" * 70)
        for s in self.steps:
            print(f"[{s.n}] 생각: {s.thought[:150]}")
            print(f"     행동: {s.action}({json.dumps(s.args, ensure_ascii=False)})")
            obs = s.observation.replace("\n", " ")
            print(f"     관찰: {'OK  ' if s.ok else '실패 '}{obs[:max_obs]}")
            print()
        print("=" * 70)
        print(f"종료 사유: {self.stop_reason}  |  스텝 {len(self.steps)}  |  실패 {self.n_failed}")
        print(f"최종 답변: {self.final_answer}")

    def to_dict(self) -> dict[str, Any]:
        return {
            "goal": self.goal,
            "steps": [
                {"n": s.n, "thought": s.thought, "action": s.action,
                 "args": s.args, "ok": s.ok, "observation": s.observation[:500]}
                for s in self.steps
            ],
            "final_answer": self.final_answer,
            "stop_reason": self.stop_reason,
            "success": self.success,
        }


# --------------------------------------------------------------------------
# 프롬프트 / Prompt
# --------------------------------------------------------------------------
REACT_SYSTEM = """\
너는 통계 분석 에이전트다. 도구를 하나씩 사용해 사용자의 질문에 답한다.

절대 규칙:
- 너는 계산하지 않는다. 평균, 표준편차, p값은 반드시 도구로 구한다.
- 한 번에 도구 하나만 부른다.
- 도구가 돌려준 값만 사실로 인정한다. 추측한 숫자를 말하지 않는다.

매 단계마다 아래 형식의 JSON 하나만 출력한다. 다른 말은 하지 않는다.

행동할 때:
{"thought": "왜 이 도구가 필요한지 한 문장", "action": "도구이름", "args": {...}}

답이 충분히 모였을 때:
{"thought": "왜 끝내는지 한 문장", "action": "finish", "args": {"answer": "최종 답변"}}

권장 순서:
1. list_columns 로 실제 열 이름과 결측 상황을 먼저 확인한다.
2. 필요하면 describe_column / check_normality 로 데이터를 진단한다.
3. 적절한 검정 도구를 부른다.
4. finish 로 결과를 해석해 답한다. p값과 효과크기를 함께 언급한다."""


def build_prompt(goal: str, toolbox: Any, steps: list[Step], max_steps: int) -> str:
    """이번 단계에 보낼 user 메시지를 만든다.

    EN — Build the user message to send for this step.
    """
    parts = [
        f"## 사용 가능한 도구\n{toolbox.describe()}\n",
        f'## 질문\n{goal}\n',
    ]

    if steps:
        parts.append("## 지금까지의 진행")
        for s in steps:
            parts.append(f"[{s.n}] 생각: {s.thought}")
            parts.append(f"    행동: {s.action}({json.dumps(s.args, ensure_ascii=False)})")
            parts.append(f"    관찰: {s.observation}")
        parts.append("")

    남은 = max_steps - len(steps)
    if 남은 <= 2:
        parts.append(
            f"** 남은 단계가 {남은}번뿐이다. 지금까지 얻은 정보로 finish 하라. **\n"
        )
    parts.append("다음에 할 일을 JSON 하나로 출력하라.")
    return "\n".join(parts)


# --------------------------------------------------------------------------
# 에이전트 / The agent
# --------------------------------------------------------------------------
class ReActAgent:
    """Thought → Action → Observation 을 반복한다.

    EN — Repeats Thought → Action → Observation until it is done.

    Parameters
    ----------
    llm : LLM 인스턴스 / an LLM instance
    toolbox : ToolRegistry (보통 StatToolbox) / usually a StatToolbox
    max_steps : 스텝 상한. 무한 루프 방지
                EN — step ceiling, to prevent an infinite loop
    max_consecutive_failures : 연속 실패가 이만큼이면 포기
                EN — give up after this many consecutive failures
    detect_repeat : 같은 (도구, 인자) 를 반복하면 경고를 관찰로 넣는다
                EN — if the same (tool, args) repeats, feed a warning back as
                the observation
    verbose : 진행 상황 출력 / print progress
    """

    def __init__(
        self,
        llm: Any,
        toolbox: Any,
        *,
        max_steps: int = 8,
        max_consecutive_failures: int = 3,
        detect_repeat: bool = True,
        verbose: bool = True,
        system: str = REACT_SYSTEM,
    ) -> None:
        self.llm = llm
        self.toolbox = toolbox
        self.max_steps = max_steps
        self.max_consecutive_failures = max_consecutive_failures
        self.detect_repeat = detect_repeat
        self.verbose = verbose
        self.system = system

    # ----------------------------------------------------------------
    def run(self, goal: str, **kw: Any) -> Trace:
        """목표를 받아 끝까지 돌린다. **예외를 밖으로 던지지 않는다.**

        EN — Take a goal and run to completion. IT NEVER RAISES.
        """
        from structured import JSONParseError, parse_json

        trace = Trace(goal=goal)
        연속실패 = 0
        본적있는행동: set[str] = set()

        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 600)

        while len(trace.steps) < self.max_steps:
            n = len(trace.steps) + 1
            step = Step(n=n)

            # --- 1) 생각하고 행동을 정한다 / Think and choose an action ---
            prompt = build_prompt(goal, self.toolbox, trace.steps, self.max_steps)
            try:
                step.raw = self.llm.chat(
                    [{"role": "system", "content": self.system},
                     {"role": "user", "content": prompt}], **kw
                )
            except Exception as exc:
                step.error = f"모델 호출 실패: {type(exc).__name__}: {exc}"
                step.observation = step.error
                trace.steps.append(step)
                연속실패 += 1
                if 연속실패 >= self.max_consecutive_failures:
                    trace.stop_reason = "모델 호출이 계속 실패함"
                    break
                continue

            try:
                결정 = parse_json(step.raw)
                if not isinstance(결정, dict):
                    raise JSONParseError("최상위가 객체가 아닙니다.")
            except JSONParseError as exc:
                step.error = str(exc)
                step.observation = (
                    f"[형식 오류] 응답을 JSON으로 읽을 수 없다: {exc}\n"
                    '{"thought": ..., "action": ..., "args": {...}} 형식으로만 출력하라.'
                )
                trace.steps.append(step)
                연속실패 += 1
                if self.verbose:
                    print(f"[{n}] 형식 오류 -> 되먹임")
                if 연속실패 >= self.max_consecutive_failures:
                    trace.stop_reason = "형식 오류가 계속됨"
                    break
                continue

            step.thought = str(결정.get("thought", ""))[:400]
            step.action = 결정.get("action")
            step.args = 결정.get("args") or {}
            if not isinstance(step.args, dict):
                step.args = {}

            if self.verbose:
                print(f"[{n}] 생각: {step.thought[:90]}")
                print(f"     행동: {step.action}({json.dumps(step.args, ensure_ascii=False)[:110]})")

            # --- 2) 종료 행동인가 / Is this the stop action? --------------
            if step.action == "finish":
                step.ok = True
                step.observation = "(종료)"
                trace.steps.append(step)
                trace.final_answer = str(step.args.get("answer", "")).strip()
                trace.stop_reason = "모델이 finish 를 선택"
                trace.success = bool(trace.final_answer)
                if self.verbose:
                    print(f"     -> 종료\n")
                break

            # --- 3) 같은 행동을 반복하는가 / Is it repeating itself? ------
            열쇠 = f"{step.action}|{json.dumps(step.args, ensure_ascii=False, sort_keys=True)}"
            if self.detect_repeat and 열쇠 in 본적있는행동:
                step.ok = False
                step.observation = (
                    "[반복 감지] 방금 전과 똑같은 도구를 똑같은 인자로 다시 불렀다. "
                    "이미 그 결과는 위에 있다. 다른 도구를 쓰거나 finish 하라."
                )
                trace.steps.append(step)
                연속실패 += 1
                if self.verbose:
                    print("     -> 반복 감지\n")
                if 연속실패 >= self.max_consecutive_failures:
                    trace.stop_reason = "같은 행동을 반복함"
                    break
                continue
            본적있는행동.add(열쇠)

            # --- 4) 도구를 실행하고 관찰한다 / Run the tool and observe ---
            result = self.toolbox.call(step.action, step.args)
            step.ok = result.ok
            step.observation = result.observation()
            step.error = result.error
            trace.steps.append(step)

            if self.verbose:
                print(f"     관찰: {'OK' if step.ok else '실패'} "
                      f"{step.observation.replace(chr(10), ' ')[:110]}\n")

            연속실패 = 0 if step.ok else 연속실패 + 1
            if 연속실패 >= self.max_consecutive_failures:
                trace.stop_reason = f"도구 실행이 {연속실패}회 연속 실패"
                break

        # --- 루프를 다 돌았는데 finish 가 없었다면 -----------------------
        # EN — the loop ended without the model ever choosing finish
        if trace.final_answer is None and not trace.stop_reason:
            trace.stop_reason = f"스텝 상한({self.max_steps}) 도달"

        if trace.final_answer is None:
            trace.final_answer = self._force_answer(trace, **kw)

        return trace

    # ----------------------------------------------------------------
    def _force_answer(self, trace: Trace, **kw: Any) -> str:
        """finish 없이 끝났을 때, 지금까지의 관찰로 답을 만들어 본다.

        빈손으로 끝내는 것보다는 낫지만, **성공으로 치지는 않는다.**
        `trace.success` 는 False 로 남는다.

        EN — When the run ended without finish, try to compose an answer from
        the observations gathered so far. Better than coming back empty-handed,
        but it DOES NOT COUNT AS SUCCESS: `trace.success` stays False.
        """
        성공관찰 = [s.observation for s in trace.steps if s.ok]
        if not 성공관찰:
            return "(도구 실행에 모두 실패해 답할 수 없습니다.)"

        모음 = "\n\n".join(성공관찰)[:3000]
        try:
            return self.llm.ask(
                f"질문: {trace.goal}\n\n"
                f"아래는 도구 실행 결과다. 이것만 근거로 답하라. "
                f"근거가 부족하면 부족하다고 말하라. 숫자를 지어내지 마라.\n\n{모음}",
                system="너는 통계 분석 결과를 해석하는 도구다.",
                temperature=0.0,
                max_tokens=kw.get("max_tokens", 600),
            )
        except Exception as exc:
            return f"(답변 생성 실패: {type(exc).__name__})"


In [ ]:
%%writefile profiler.py
"""전산실험2 9주차 : 데이터 진단과 분석계획 수립.

7주차까지 우리는 **깨끗한 데이터를 전제**했다. 실제 데이터는 그렇지 않다.
`-999` 가 결측이고, `'12.5kg'` 이 수치이고, `Yes/yes/Y` 가 같은 값이다.

이 모듈은 두 가지를 한다.

    DataProfiler   데이터를 훑어 타입을 추론하고 문제를 찾아낸다 (규칙 기반)
    AnalysisPlanner LLM이 분석 계획을 세우되, **규칙이 검증한다**

핵심 설계 원칙:
    **무엇을 LLM에게 맡기고 무엇을 규칙으로 고정할 것인가.**

    규칙으로 고정  : 검정의 전제 조건 (집단 수, 대응 여부, 척도)
                    -> 틀리면 결과 자체가 무의미하므로 코드가 강제한다
    LLM에게 위임   : 변수의 의미 파악, 사용자 질문 해석, 결과 서술
                    -> 규칙으로 적을 수 없는 판단

EN — Week 9: data diagnosis and building an analysis plan.
Up to week 7 we ASSUMED CLEAN DATA. Real data is not: `-999` means missing,
`'12.5kg'` is a number, and `Yes/yes/Y` are one value. This module does two
things: DataProfiler scans the data, infers types and finds problems (all
rule-based), and AnalysisPlanner lets the LLM draft an analysis plan WHICH THE
RULES THEN VERIFY.
The key design question is what to delegate to the LLM and what to pin down in
code. Pinned in rules: the preconditions of each test (number of groups, paired
or not, measurement scale) — get those wrong and the result is meaningless, so
the code enforces them. Delegated to the LLM: reading what the variables mean,
interpreting the user's question, describing the result — judgements you cannot
write as rules.

사용 예 · Example:
    import pandas as pd
    from profiler import DataProfiler

    prof = DataProfiler(pd.read_csv("exam_greenhouse.csv"))
    print(prof.report())
    clean = prof.clean()
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Any

__all__ = [
    "SENTINELS",
    "ColumnProfile",
    "DataProfiler",
    "TEST_RULES",
    "validate_plan",
    "AnalysisPlanner",
]

# 결측을 뜻하는 관례적 값들. 센서·설문 데이터에서 흔하다.
# EN — Conventional stand-ins for "missing"; common in sensor and survey data.
SENTINELS = {-999, -9999, -99, 999, 9999, -1e30}
SENTINEL_STRINGS = {"", "na", "n/a", "null", "none", "nan", "-", "?", "결측", "무응답"}

_UNIT_SUFFIX = re.compile(r"^\s*([-+]?[\d.,]+)\s*[a-zA-Z가-힣%°]+\s*$")
_DECIMAL_COMMA = re.compile(r"^\s*[-+]?\d{1,3}(,\d{1,2})\s*$")

# 같은 뜻인데 표기가 다른 값들. 설문·수기 입력에서 늘 나온다.
# 축약형(y/n)과 완전형(yes/no)이 한 열에 섞이면 대소문자 정규화만으로는
# 합쳐지지 않는다.
# EN — Same meaning, different spelling; unavoidable in surveys and hand entry.
#      When a column mixes the short form (y/n) with the full form (yes/no),
#      case normalisation alone will not merge them.
SYNONYM_FAMILIES: list[dict[str, str]] = [
    {"y": "yes", "yes": "yes", "true": "yes", "t": "yes", "1": "yes",
     "n": "no", "no": "no", "false": "no", "f": "no", "0": "no",
     "예": "yes", "아니오": "no", "아니요": "no"},
    {"m": "male", "male": "male", "남": "male", "남자": "male", "남성": "male",
     "f": "female", "female": "female", "여": "female", "여자": "female",
     "여성": "female"},
]


# --------------------------------------------------------------------------
# 열 하나의 진단 결과 / The diagnosis of a single column
# --------------------------------------------------------------------------
@dataclass
class ColumnProfile:
    name: str
    dtype: str
    n: int
    n_missing: int = 0
    n_unique: int = 0
    kind: str = "unknown"          # continuous / ordinal / nominal / id / binary
    issues: list[str] = field(default_factory=list)
    fixes: list[str] = field(default_factory=list)   # 적용 가능한 정리 방법 / cleanups that apply here
    stats: dict[str, Any] = field(default_factory=dict)
    levels: list[str] = field(default_factory=list)

    def line(self) -> str:
        문제 = f"  ⚠ {'; '.join(self.issues)}" if self.issues else ""
        return (f"- {self.name} [{self.kind}] "
                f"n={self.n - self.n_missing}/{self.n}, 고유 {self.n_unique}{문제}")


# --------------------------------------------------------------------------
# 데이터 진단 / Data diagnosis
# --------------------------------------------------------------------------
class DataProfiler:
    """DataFrame 을 훑어 타입을 추론하고 문제를 찾아낸다.

    **전부 규칙 기반이다. LLM을 쓰지 않는다.**
    타입 추론과 결측 탐지는 규칙으로 충분히 정확하고,
    LLM에게 맡기면 느리고 비싸고 틀린다.

    EN — Scan a DataFrame, infer column types and find problems. ENTIRELY
    RULE-BASED — no LLM. Type inference and missing-value detection are accurate
    enough as rules; handing them to an LLM would be slow, expensive and wrong.
    """

    def __init__(self, df: Any, *, ordinal_max_levels: int = 10) -> None:
        self.df = df
        self.ordinal_max_levels = ordinal_max_levels
        self.profiles: dict[str, ColumnProfile] = {}
        self._scan()

    # ----------------------------------------------------------------
    def _scan(self) -> None:
        import pandas as pd

        n = len(self.df)
        for col in self.df.columns:
            s = self.df[col]
            p = ColumnProfile(name=col, dtype=str(s.dtype), n=n,
                              n_missing=int(s.isna().sum()),
                              n_unique=int(s.nunique(dropna=True)))

            문자 = s.dropna().astype(str)

            # --- 수치로 볼 수 있는가 / can this be read as numeric? ----
            숫자 = pd.to_numeric(s, errors="coerce")
            변환실패 = int(숫자.isna().sum() - s.isna().sum())
            수치비율 = 숫자.notna().sum() / max(n - p.n_missing, 1)

            # --- 문자열에 숨은 수치 패턴 / numbers hiding inside strings
            단위붙음 = int(문자.apply(lambda v: bool(_UNIT_SUFFIX.match(v))).sum())
            쉼표소수 = int(문자.apply(lambda v: bool(_DECIMAL_COMMA.match(v))).sum())

            if 단위붙음:
                p.issues.append(f"단위가 붙은 값 {단위붙음}건 (예: '12.5kg')")
                p.fixes.append("strip_units")
            if 쉼표소수:
                p.issues.append(f"소수점에 쉼표를 쓴 값 {쉼표소수}건 (예: '3,5')")
                p.fixes.append("comma_decimal")

            수치가능 = 수치비율 > 0.7 or (단위붙음 + 쉼표소수) > 0

            # --- 결측 위장값 / values that are really missing ----------
            if 수치가능:
                유효 = 숫자.dropna()
                감지 = sorted({v for v in SENTINELS if (유효 == v).any()})
                if 감지:
                    개수 = int(유효.isin(감지).sum())
                    p.issues.append(f"결측 위장값 {감지} {개수}건 (실제 결측일 가능성)")
                    p.fixes.append("sentinel_to_nan")

            빈문자 = int(문자.str.strip().str.lower().isin(SENTINEL_STRINGS).sum())
            if 빈문자:
                p.issues.append(f"결측을 뜻하는 문자열 {빈문자}건")
                p.fixes.append("string_to_nan")

            # --- 타입 판정 / decide the column type --------------------
            if p.n_unique == n and n > 10:
                p.kind = "id"
            elif 수치가능:
                유효 = 숫자[~숫자.isin(SENTINELS)].dropna()
                if len(유효) >= 3:
                    p.stats = {
                        "mean": round(float(유효.mean()), 4),
                        "sd": round(float(유효.std(ddof=1)), 4),
                        "min": round(float(유효.min()), 4),
                        "median": round(float(유효.median()), 4),
                        "max": round(float(유효.max()), 4),
                        "skew": round(float(유효.skew()), 3),
                    }
                    self._check_outliers(p, 유효)
                    self._check_normality(p, 유효)
                # 정수이고 수준이 적으면 순서형일 수 있다
                # EN — integer with few levels may well be ordinal
                정수형 = bool((유효 % 1 == 0).all()) if len(유효) else False
                if 정수형 and p.n_unique <= self.ordinal_max_levels:
                    p.kind = "ordinal"
                    p.issues.append(
                        f"정수 {p.n_unique}수준 — 순서형일 수 있음 "
                        f"(리커트 척도 등). 평균을 내는 것이 적절한지 확인할 것"
                    )
                else:
                    p.kind = "continuous"
                if 변환실패:
                    p.issues.append(f"수치로 못 바꾸는 값 {변환실패}건")
            else:
                정규화 = 문자.str.strip().str.lower()
                수준 = sorted(정규화.unique())
                p.levels = [str(v) for v in sorted(문자.unique())][:20]
                if len(수준) < p.n_unique:
                    p.issues.append(
                        f"대소문자·공백만 다른 값이 섞여 있음 "
                        f"(표기 {p.n_unique}종 → 실제 {len(수준)}종)"
                    )
                    p.fixes.append("normalize_case")

                # 축약형과 완전형이 섞였는가 (y / yes)
                # EN — are short and full forms mixed together?
                family = self._match_synonym_family(수준)
                if family is not None:
                    합친수준 = sorted({family[v] for v in 수준})
                    if len(합친수준) < len(수준):
                        p.issues.append(
                            f"같은 뜻의 다른 표기가 섞여 있음 "
                            f"({수준} → {합친수준})"
                        )
                        p.fixes.append("merge_synonyms")
                        수준 = 합친수준
                p.kind = "binary" if len(수준) == 2 else "nominal"

            # --- 결측률 / missing rate ---------------------------------
            결측률 = p.n_missing / n if n else 0
            if 결측률 > 0.3:
                p.issues.append(f"결측률 {결측률:.0%} — 분석에서 제외를 고려")
            elif 결측률 > 0:
                p.issues.append(f"결측 {p.n_missing}건 ({결측률:.1%})")

            self.profiles[col] = p

    # ----------------------------------------------------------------
    @staticmethod
    def _match_synonym_family(levels: list[str]) -> dict[str, str] | None:
        """이 열의 값들이 알려진 동의어 집합에 전부 들어가는가.

        일부만 걸치면 `None` 을 돌려준다. 예를 들어 값이
        `['y', 'n', 'maybe']` 라면 함부로 합치지 않는다 —
        모르는 값이 섞여 있으면 손대지 않는 것이 안전하다.

        EN — Do ALL of this column's values fall inside one known synonym
        family? A partial match returns `None`: with values like
        `['y', 'n', 'maybe']` we refuse to merge, because leaving an unknown
        value alone is the safe choice.
        """
        for family in SYNONYM_FAMILIES:
            if all(v in family for v in levels):
                return family
        return None

    @staticmethod
    def _check_outliers(p: ColumnProfile, values: Any) -> None:
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr <= 0:
            return
        low, high = q1 - 3 * iqr, q3 + 3 * iqr      # 3배 = 극단값만 / 3×IQR: extreme values only
        극단 = values[(values < low) | (values > high)]
        if len(극단):
            p.issues.append(
                f"극단 이상치 {len(극단)}건 "
                f"(예: {[round(float(v), 1) for v in 극단.head(3)]}) — 입력 오류 가능"
            )
            p.stats["outlier_bounds"] = [round(float(low), 2), round(float(high), 2)]

    @staticmethod
    def _check_normality(p: ColumnProfile, values: Any) -> None:
        if len(values) < 3:
            return
        try:
            from scipy import stats

            pv = float(stats.shapiro(values[:5000]).pvalue)
            p.stats["shapiro_p"] = pv
            p.stats["normal_at_0.05"] = bool(pv >= 0.05)
        except Exception:
            pass

    # ----------------------------------------------------------------
    def report(self) -> str:
        """사람이 읽는 진단 보고서. / The diagnosis report, for a human to read."""
        줄 = [f"데이터: {len(self.df)}행 {len(self.df.columns)}열", ""]
        for kind in ["id", "continuous", "ordinal", "binary", "nominal", "unknown"]:
            해당 = [p for p in self.profiles.values() if p.kind == kind]
            if not 해당:
                continue
            줄.append(f"[{kind}] {len(해당)}개")
            줄 += [p.line() for p in 해당]
            줄.append("")
        문제열 = [p.name for p in self.profiles.values() if p.fixes]
        if 문제열:
            줄.append(f"정리가 필요한 열: {문제열}")
        return "\n".join(줄)

    def to_prompt(self, max_cols: int = 30) -> str:
        """LLM 프롬프트에 넣을 압축 요약.

        `report()` 보다 짧다. 토큰 예산 때문이다 (4주차).

        EN — A compressed summary for the LLM prompt. Shorter than `report()`,
        because of the token budget (week 4).
        """
        줄 = [f"행 {len(self.df)}개, 열 {len(self.df.columns)}개"]
        for p in list(self.profiles.values())[:max_cols]:
            부분 = f"- {p.name}: {p.kind}"
            if p.kind in ("binary", "nominal") and p.levels:
                부분 += f" {p.levels[:6]}"
            if p.stats.get("mean") is not None:
                부분 += f" (평균 {p.stats['mean']}, 결측 {p.n_missing})"
            if p.stats.get("normal_at_0.05") is False:
                부분 += " [정규성 기각]"
            if p.issues:
                부분 += f"  ※ {p.issues[0]}"
            줄.append(부분)
        return "\n".join(줄)

    def issues(self) -> dict[str, list[str]]:
        return {n: p.issues for n, p in self.profiles.items() if p.issues}

    # ----------------------------------------------------------------
    def clean(self, *, drop_outliers: bool = False, verbose: bool = True) -> Any:
        """진단에서 찾은 문제를 실제로 고친 DataFrame 을 돌려준다.

        **원본을 바꾸지 않는다.** 무엇을 고쳤는지 출력한다 —
        조용히 데이터를 바꾸는 것은 통계 분석에서 가장 위험한 일이다.

        EN — Return a DataFrame with the diagnosed problems actually fixed. IT
        DOES NOT MODIFY THE ORIGINAL, and it prints what it changed: silently
        altering data is the single most dangerous thing you can do in a
        statistical analysis.
        """
        import numpy as np
        import pandas as pd

        out = self.df.copy()
        기록 = []

        for name, p in self.profiles.items():
            s = out[name]

            if "comma_decimal" in p.fixes:
                s = s.astype(str).str.replace(
                    r"^(\s*[-+]?\d{1,3}),(\d{1,2}\s*)$", r"\1.\2", regex=True)
                기록.append(f"{name}: 소수점 쉼표 → 마침표")

            if "strip_units" in p.fixes:
                s = s.astype(str).str.replace(r"\s*[a-zA-Z가-힣%°]+\s*$", "", regex=True)
                기록.append(f"{name}: 단위 문자 제거")

            if "string_to_nan" in p.fixes:
                s = s.replace(
                    {v: np.nan for v in SENTINEL_STRINGS}
                ).replace(r"^\s*$", np.nan, regex=True)
                기록.append(f"{name}: 결측 문자열 → NaN")

            if "normalize_case" in p.fixes:
                s = s.astype(str).str.strip().str.lower()
                기록.append(f"{name}: 대소문자·공백 정규화")

            if "merge_synonyms" in p.fixes:
                정규 = s.astype(str).str.strip().str.lower()
                family = self._match_synonym_family(sorted(정규.dropna().unique()))
                if family is not None:
                    전 = sorted(정규.unique())
                    s = 정규.map(family)
                    기록.append(f"{name}: 동의어 통합 {전} → {sorted(s.dropna().unique())}")

            if p.kind in ("continuous", "ordinal"):
                s = pd.to_numeric(s, errors="coerce")
                if "sentinel_to_nan" in p.fixes:
                    before = int(s.notna().sum())
                    s = s.mask(s.isin(SENTINELS))
                    기록.append(f"{name}: 결측 위장값 → NaN ({before - int(s.notna().sum())}건)")

                if drop_outliers and "outlier_bounds" in p.stats:
                    low, high = p.stats["outlier_bounds"]
                    before = int(s.notna().sum())
                    s = s.mask((s < low) | (s > high))
                    지운수 = before - int(s.notna().sum())
                    if 지운수:
                        기록.append(f"{name}: 극단 이상치 {지운수}건 → NaN")

            out[name] = s

        if verbose:
            if 기록:
                print("적용한 정리:")
                for r in 기록:
                    print(f"  - {r}")
            else:
                print("정리할 것이 없습니다.")
            if not drop_outliers:
                이상 = [n for n, p in self.profiles.items() if "outlier_bounds" in p.stats]
                if 이상:
                    print(f"\n[주의] 이상치가 있는 열: {이상}")
                    print("       drop_outliers=True 로 제거할 수 있으나,")
                    print("       제거 여부는 분석자가 판단할 일입니다.")
        return out


# --------------------------------------------------------------------------
# 검정 선택 규칙 — 도메인 지식을 코드로 고정한다
# EN — Test-selection rules: domain knowledge pinned down in code
# --------------------------------------------------------------------------
TEST_RULES: dict[str, dict[str, Any]] = {
    "paired-t-test": {
        "설명": "같은 대상의 두 시점 비교, 차이가 정규",
        "outcome": ["continuous"],
        "design": "paired",
        "n_groups": 2,
        "normality_required": True,
    },
    "wilcoxon-signed-rank": {
        "설명": "같은 대상의 두 시점 비교, 정규성 불만족",
        "outcome": ["continuous", "ordinal"],
        "design": "paired",
        "n_groups": 2,
        "normality_required": False,
    },
    "independent-t-test": {
        "설명": "서로 다른 두 집단 비교, 정규",
        "outcome": ["continuous"],
        "design": "independent",
        "n_groups": 2,
        "normality_required": True,
    },
    "mann-whitney-u": {
        "설명": "서로 다른 두 집단 비교, 정규성 불만족",
        "outcome": ["continuous", "ordinal"],
        "design": "independent",
        "n_groups": 2,
        "normality_required": False,
    },
    "one-way-anova": {
        "설명": "세 집단 이상 비교, 정규 + 등분산",
        "outcome": ["continuous"],
        "design": "independent",
        "n_groups": "3+",
        "normality_required": True,
    },
    "kruskal-wallis": {
        "설명": "세 집단 이상 비교, 정규성 불만족",
        "outcome": ["continuous", "ordinal"],
        "design": "independent",
        "n_groups": "3+",
        "normality_required": False,
    },
    "chi-square": {
        "설명": "두 범주형 변수의 연관성",
        "outcome": ["nominal", "binary"],
        "design": "independent",
        "n_groups": "any",
        "normality_required": False,
    },
    "pearson-correlation": {
        "설명": "두 연속형 변수의 선형 관계, 정규",
        "outcome": ["continuous"],
        "design": "correlation",
        "n_groups": None,
        "normality_required": True,
    },
    "spearman-correlation": {
        "설명": "두 변수의 단조 관계, 정규성 불만족 또는 순서형",
        "outcome": ["continuous", "ordinal"],
        "design": "correlation",
        "n_groups": None,
        "normality_required": False,
    },
}


def validate_plan(plan: dict[str, Any], profiler: DataProfiler) -> list[str]:
    """LLM이 세운 분석 계획을 **규칙으로 검증**한다.

    이것이 이 모듈의 핵심이다. LLM은 "이 질문에는 t검정이 맞겠다" 까지만
    하고, 그것이 **데이터의 실제 성질과 맞는지는 코드가 확인**한다.

    EN — Validate the LLM's analysis plan AGAINST THE RULES. This is the point
    of the module: the LLM gets as far as "a t-test suits this question", and
    THE CODE checks whether that matches what the data actually is.

    계획 형식 · Plan format::

        {
            "test": "independent-t-test",
            "outcome_column": "yield_after",
            "group_column": "fertilizer",     # 선택
            "second_column": "water_liters",  # 상관분석용, 선택
            "design": "independent",
            "reason": "...",
        }
    """
    문제: list[str] = []
    test = plan.get("test")

    if test not in TEST_RULES:
        return [f"'{test}' 는 알 수 없는 검정입니다. 가능: {sorted(TEST_RULES)}"]

    rule = TEST_RULES[test]
    prof = profiler.profiles

    # --- 결과변수 / the outcome variable ---------------------------
    outcome = plan.get("outcome_column")
    if not outcome:
        문제.append("outcome_column 이 없습니다.")
    elif outcome not in prof:
        문제.append(f"'{outcome}' 열이 데이터에 없습니다. 있는 열: {list(prof)}")
    else:
        kind = prof[outcome].kind
        if kind not in rule["outcome"]:
            문제.append(
                f"{test} 는 결과변수가 {rule['outcome']} 이어야 하는데 "
                f"'{outcome}' 는 {kind} 입니다."
            )
        if kind == "id":
            문제.append(f"'{outcome}' 는 식별자로 보입니다. 분석 대상이 아닙니다.")

    # --- 집단변수와 집단 수 / group variable and group count -------
    # 대응표본은 '집단 열'이 아니라 '두 개의 측정 열'을 쓴다.
    # n_groups=2 라는 표기에 낚여 group_column 을 요구하면 안 된다.
    # EN — A paired design uses TWO MEASUREMENT COLUMNS, not a group column.
    #      Do not be fooled by n_groups=2 into demanding a group_column.
    group = plan.get("group_column")
    집단열_필요 = rule["design"] == "independent" and rule["n_groups"] in (2, "3+")
    if 집단열_필요:
        if not group:
            문제.append(f"{test} 에는 group_column 이 필요합니다.")
        elif group not in prof:
            문제.append(f"'{group}' 열이 데이터에 없습니다.")
        else:
            수준수 = prof[group].n_unique
            if rule["n_groups"] == 2 and 수준수 != 2:
                대안 = "one-way-anova / kruskal-wallis" if 수준수 > 2 else "(수준이 1개)"
                문제.append(
                    f"{test} 는 두 집단 비교인데 '{group}' 의 수준이 {수준수}개입니다. "
                    f"→ {대안} 를 고려하세요."
                )
            if rule["n_groups"] == "3+" and 수준수 < 3:
                문제.append(
                    f"{test} 는 세 집단 이상인데 '{group}' 의 수준이 {수준수}개입니다. "
                    f"→ independent-t-test / mann-whitney-u 를 고려하세요."
                )
            if prof[group].kind == "continuous":
                문제.append(
                    f"'{group}' 는 연속형입니다. 집단 변수로 쓰려면 범주화가 필요합니다."
                )

    # --- 대응 설계 / paired design ---------------------------------
    design = plan.get("design")
    if design and design != rule["design"]:
        문제.append(
            f"{test} 는 {rule['design']} 설계용인데 계획에는 {design} 이라고 되어 있습니다."
        )
    if rule["design"] == "paired" and not plan.get("second_column"):
        문제.append(
            "대응표본 검정에는 두 번째 측정 열(second_column)이 필요합니다."
        )

    # --- 상관분석 / correlation ------------------------------------
    if rule["design"] == "correlation":
        second = plan.get("second_column")
        if not second:
            문제.append("상관분석에는 second_column 이 필요합니다.")
        elif second not in prof:
            문제.append(f"'{second}' 열이 데이터에 없습니다.")
        elif prof[second].kind not in rule["outcome"]:
            문제.append(
                f"{test} 는 두 변수가 모두 {rule['outcome']} 이어야 하는데 "
                f"'{second}' 는 {prof[second].kind} 입니다."
            )

    # --- 정규성 / normality ----------------------------------------
    if rule["normality_required"] and outcome in prof:
        정규 = prof[outcome].stats.get("normal_at_0.05")
        if 정규 is False:
            비모수 = {
                "paired-t-test": "wilcoxon-signed-rank",
                "independent-t-test": "mann-whitney-u",
                "one-way-anova": "kruskal-wallis",
                "pearson-correlation": "spearman-correlation",
            }.get(test, "비모수 검정")
            문제.append(
                f"'{outcome}' 는 정규성 검정에서 기각되었습니다 "
                f"(Shapiro p={prof[outcome].stats.get('shapiro_p', 0):.2e}). "
                f"→ {비모수} 를 고려하세요."
            )

    return 문제


# --------------------------------------------------------------------------
# 분석계획 수립기 / The analysis planner
# --------------------------------------------------------------------------
PLANNER_SYSTEM = """\
너는 통계 분석 계획을 세우는 도구다. 데이터 요약과 사용자 질문을 받아
어떤 검정을 어떤 변수로 수행할지 정한다.

너는 계산하지 않는다. 계획만 세운다.
데이터에 실제로 있는 열 이름만 쓴다. 없는 열을 지어내지 않는다.
확신이 없으면 reason 에 그렇게 적는다."""


class AnalysisPlanner:
    """LLM이 계획을 세우고, 규칙이 검증하고, 실패하면 되먹인다.

    3주차 `ask_json()` 과 같은 구조다. 달라진 것은 검증이
    **스키마 검사가 아니라 통계적 타당성 검사**라는 점이다.

    EN — The LLM drafts the plan, the rules verify it, and failures are fed
    back. Same structure as `ask_json()` in week 3; what changed is that the
    check is STATISTICAL VALIDITY rather than schema conformance.
    """

    def __init__(self, llm: Any, profiler: DataProfiler, *, n_retry: int = 3,
                 verbose: bool = True) -> None:
        self.llm = llm
        self.profiler = profiler
        self.n_retry = n_retry
        self.verbose = verbose

    def plan(self, question: str, **kw: Any) -> dict[str, Any]:
        """질문에 대한 분석 계획을 세운다.

        EN — Build an analysis plan for the question.

        Returns
        -------
        dict : 검증을 통과한 계획.
               실패하면 {"test": None, "error": ..., "attempts": [...]}
               EN — the validated plan, or the error dict above on failure
        """
        from structured import JSONParseError, parse_json

        schema_desc = {
            "test": f"다음 중 하나: {sorted(TEST_RULES)}",
            "outcome_column": "분석할 결과변수 열 이름",
            "group_column": "집단 비교면 집단 열 이름, 아니면 null",
            "second_column": "대응표본의 두 번째 측정 열 또는 상관분석의 두 번째 변수, 아니면 null",
            "design": "paired / independent / correlation 중 하나",
            "reason": "이 검정을 고른 이유 한두 문장",
        }
        규칙설명 = "\n".join(
            f"- {k}: {v['설명']} (결과변수 {v['outcome']}, 설계 {v['design']}, 집단 {v['n_groups']})"
            for k, v in TEST_RULES.items()
        )

        base = (
            f"## 데이터 요약\n{self.profiler.to_prompt()}\n\n"
            f"## 선택 가능한 검정\n{규칙설명}\n\n"
            f"## 질문\n{question}\n\n"
            f"## 출력 형식 (JSON만)\n"
            + "\n".join(f'  "{k}": {v}' for k, v in schema_desc.items())
            + "\n\n설명이나 코드블록 없이 JSON 객체 하나만 출력하라."
        )

        messages = [
            {"role": "system", "content": PLANNER_SYSTEM},
            {"role": "user", "content": base},
        ]
        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 500)
        시도들 = []

        for attempt in range(1, self.n_retry + 1):
            try:
                원문 = self.llm.chat(messages, **kw)
            except Exception as exc:
                # 서버 문제로 루프 전체가 죽으면 안 된다 (6주차와 같은 원칙)
                # EN — a server problem must not kill the whole loop (week 6)
                시도들.append({"attempt": attempt, "error": f"{type(exc).__name__}: {exc}"})
                if self.verbose:
                    print(f"[{attempt}회] 모델 호출 실패: {type(exc).__name__}")
                if attempt == self.n_retry:
                    return {"test": None,
                            "error": f"모델 호출 실패: {type(exc).__name__}: {exc}",
                            "attempts": 시도들}
                continue

            try:
                계획 = parse_json(원문)
            except JSONParseError as exc:
                시도들.append({"attempt": attempt, "error": str(exc)})
                피드백 = f"JSON으로 읽을 수 없었습니다: {exc}\nJSON 객체만 출력하세요."
            else:
                문제 = validate_plan(계획, self.profiler)
                시도들.append({"attempt": attempt, "plan": 계획, "problems": 문제})
                if not 문제:
                    if self.verbose:
                        print(f"[계획 수립 성공 - {attempt}회] {계획.get('test')}")
                    계획["attempts"] = 시도들
                    return 계획
                if self.verbose:
                    print(f"[{attempt}회] {계획.get('test')} → 검증 실패: {문제[0]}")
                피드백 = (
                    "이 계획은 데이터의 성질과 맞지 않습니다:\n"
                    + "\n".join(f"- {p}" for p in 문제)
                    + "\n위 지적을 반영해 계획을 고쳐 JSON만 다시 출력하세요."
                )

            messages.append({"role": "assistant", "content": 원문})
            messages.append({"role": "user", "content": 피드백})

        return {
            "test": None,
            "error": f"{self.n_retry}회 시도 모두 검증을 통과하지 못했습니다.",
            "attempts": 시도들,
        }


In [ ]:
%%writefile evaluate.py
"""전산실험2 11주차 : 에이전트 평가 — 성능을 추정 문제로 다룬다.

지금까지 매주 성공률을 재기는 했다. 그런데 n=10, n=20 수준이었고
신뢰구간이 너무 넓어 사실 아무것도 주장할 수 없었다.

이 모듈은 그것을 제대로 한다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 수업이 다른 AI 강의와 갈리는 지점

 에이전트를 k번 반복 실행해 얻은 관측은 **서로 독립이 아니다.**
 같은 문항을 여러 번 돌린 것이기 때문이다. 어떤 문항은 늘 맞고
 어떤 문항은 늘 틀린다. 문항 안의 관측은 서로 닮아 있다.

 그런데 대부분의 AI 평가 코드는 이것을 무시하고
 (맞은 횟수) / (전체 시행) 을 그냥 이항비율로 다룬다.
 → **신뢰구간이 실제보다 좁게 나온다. 없는 차이를 있다고 말하게 된다.**

 이것은 통계학에서 오래 알려진 **군집자료(clustered data)** 문제다.
 여러분은 이것을 아는 사람들이다. 제대로 하자.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

구성:
    GoldenItem / GoldenSet   정답이 정해진 평가 문항
    채점 함수들               keyword / numeric / choice / judge
    run_eval                 반복 실행해 결과를 모은다
    summarize                정확도 + 신뢰구간 (군집 보정 포함)
    compare_conditions       두 조건을 짝지어 비교 (McNemar)
    required_n               표본 크기 계산

EN — Week 11: evaluating an agent — treating performance as an estimation
problem. We have measured success rates every week, but at n=10 or n=20 the
confidence interval was so wide that nothing could actually be claimed.

WHERE THIS COURSE PARTS WAYS WITH OTHER AI CLASSES: the observations you get by
running an agent k times ARE NOT INDEPENDENT — they are repeats of the same
items. Some items are always right and some always wrong, so observations
within an item resemble each other. Most AI evaluation code ignores this and
treats (number correct)/(number of trials) as a plain binomial proportion,
which makes THE CONFIDENCE INTERVAL NARROWER THAN IT REALLY IS and leads you to
claim a difference that is not there. This is the long-known problem of
CLUSTERED DATA in statistics. You are the people who know this. Let's do it
properly.

The pieces: GoldenItem/GoldenSet (evaluation items with known answers), the
scorers (keyword / numeric / choice / judge), run_eval (run repeatedly and
collect), summarize (accuracy plus confidence intervals, cluster-corrected),
compare_conditions (paired comparison of two conditions, McNemar) and
required_n (sample-size calculation).

사용 예 · Example:
    from evaluate import GoldenItem, run_eval, summarize

    items = [GoldenItem("q1", "치료 전후 차이가 있나?", expected="유의",
                        scorer="keyword")]
    df = run_eval(items, lambda q: agent.run(q).final_answer, repeats=5)
    print(summarize(df))
"""

from __future__ import annotations

import math
import re
from dataclasses import dataclass, field
from typing import Any, Callable

__all__ = [
    "GoldenItem",
    "score_keyword",
    "score_numeric",
    "score_choice",
    "make_judge",
    "run_eval",
    "summarize",
    "per_item",
    "consistency",
    "compare_conditions",
    "required_n",
    "wilson",
]


# --------------------------------------------------------------------------
# 평가 문항 / Evaluation items
# --------------------------------------------------------------------------
@dataclass
class GoldenItem:
    """정답이 정해진 평가 문항 하나.

    EN — One evaluation item with a known correct answer. `expected` takes a
    different shape per scorer: a string or list of strings for keyword (all
    must appear), a float for numeric, one of the allowed strings for choice.

    Parameters
    ----------
    id       문항 식별자
    question 에이전트에게 줄 질문
    expected 정답. 채점 방식에 따라 형태가 다르다
             keyword -> str 또는 list[str] (모두 포함되어야 함)
             numeric -> float
             choice  -> str (허용 목록 중 하나)
    scorer   "keyword" | "numeric" | "choice" | callable
    tolerance numeric 채점의 상대 허용오차
    tags     분석용 꼬리표 (난이도, 유형 등)
    """

    id: str
    question: str
    expected: Any
    scorer: Any = "keyword"
    tolerance: float = 0.01
    tags: dict[str, Any] = field(default_factory=dict)


# --------------------------------------------------------------------------
# 채점 함수 / Scorers
# --------------------------------------------------------------------------
def score_keyword(answer: str, expected: Any, **_: Any) -> bool:
    """정답 키워드가 모두 들어 있는가.

    가장 단순하고 가장 자주 쓴다. 다만 **표현이 달라지면 놓친다.**
    "유의하다" 를 기대했는데 "통계적으로 의미 있는 차이" 라고 답하면 오답 처리된다.
    이 한계를 알고 써야 한다. 대안은 `make_judge`.

    EN — Are all the expected keywords present? The simplest scorer and the one
    used most, but IT MISSES PARAPHRASES: expect "유의하다" and an answer saying
    "통계적으로 의미 있는 차이" is marked wrong. Know that limit before using it;
    the alternative is `make_judge`.
    """
    if answer is None:
        return False
    본문 = str(answer).lower()
    필요 = [expected] if isinstance(expected, str) else list(expected)
    return all(str(k).lower() in 본문 for k in 필요)


_NUM = re.compile(r"[-+]?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?")


def score_numeric(answer: str, expected: float, tolerance: float = 0.01, **_: Any) -> bool:
    """답변에 들어 있는 수치 중 하나가 정답과 충분히 가까운가.

    p값처럼 자릿수가 큰 값은 **상대오차**로 봐야 한다.
    p=1e-20 과 p=2e-20 의 절대차는 0에 가깝지만 둘은 다른 값이 아니다
    (둘 다 "매우 작다"). 반대로 0.04 와 0.06 은 절대차가 작아도
    결론이 갈린다. 여기서는 상대오차를 쓰되, 문항에서 tolerance 를 조절한다.

    EN — Is one of the numbers in the answer close enough to the expected one?
    Values spanning many orders of magnitude, such as p-values, must be judged
    by RELATIVE error: p=1e-20 and p=2e-20 differ by almost nothing in absolute
    terms and mean the same thing ("very small"), whereas 0.04 and 0.06 are
    absolutely close but lead to opposite conclusions. We use relative error and
    let each item tune `tolerance`.
    """
    if answer is None:
        return False
    후보 = []
    for m in _NUM.findall(str(answer)):
        try:
            후보.append(float(m.replace(",", "")))
        except ValueError:
            pass
    if not 후보:
        return False
    if expected == 0:
        return any(abs(v) <= tolerance for v in 후보)
    return any(abs(v - expected) / abs(expected) <= tolerance for v in 후보)


def score_choice(answer: str, expected: str, **_: Any) -> bool:
    """답변이 정답 선택지를 명시했는가. 구두점·대소문자·공백 차이를 무시한다.

    EN — Did the answer name the correct choice? Punctuation, case and spacing
    differences are ignored.
    """
    if answer is None:
        return False
    정규 = lambda s: re.sub(r"[\s\-_]", "", str(s).lower())
    return 정규(expected) in 정규(answer)


def make_judge(llm: Any, criteria: str = "", **kw: Any) -> Callable[..., bool]:
    """LLM-as-judge 채점기를 만든다.

    표현이 달라도 뜻이 맞으면 정답으로 인정하고 싶을 때 쓴다.

    **주의: 판정자도 모델이다.**
      - 판정자가 틀릴 수 있다 → 사람이 채점한 것과 일치율을 먼저 재라
      - 판정자와 피평가자가 같은 모델이면 편향이 생긴다 (자기 답을 후하게 봄)
      - 판정 자체가 확률적이다 → 판정도 여러 번 해서 다수결

    그래서 이 수업의 기본 채점은 keyword/numeric 이고,
    judge 는 **그것으로 안 되는 문항에만** 쓴다.

    EN — Build an LLM-as-judge scorer, for when a different wording of the same
    meaning should still count as correct. CAUTION — THE JUDGE IS ALSO A MODEL:
    it can be wrong (measure its agreement with human grading first); if judge
    and candidate are the same model there is bias (it marks its own answers
    generously); and the verdict itself is probabilistic (judge several times
    and take the majority). That is why the default scorers in this course are
    keyword/numeric, and `judge` is used ONLY where those cannot work.
    """
    kw.setdefault("temperature", 0.0)
    kw.setdefault("max_tokens", 200)

    SYSTEM = (
        "너는 채점자다. 학생의 답변이 정답과 같은 뜻인지 판정한다.\n"
        "표현이 달라도 내용이 맞으면 정답으로 인정한다.\n"
        "핵심 결론이 다르면 오답이다. 애매하면 오답으로 처리한다.\n"
        '반드시 {"correct": true} 또는 {"correct": false} 형태의 JSON만 출력한다.'
        + (f"\n추가 기준: {criteria}" if criteria else "")
    )

    def judge(answer: str, expected: Any, **_: Any) -> bool:
        from structured import JSONParseError, parse_json

        if answer is None:
            return False
        prompt = (f"## 정답\n{expected}\n\n## 학생 답변\n{answer}\n\n"
                  f"같은 뜻인가? JSON만 출력하라.")
        try:
            out = llm.chat([{"role": "system", "content": SYSTEM},
                            {"role": "user", "content": prompt}], **kw)
            return bool(parse_json(out).get("correct", False))
        except (JSONParseError, Exception):
            return False

    return judge


_SCORERS = {"keyword": score_keyword, "numeric": score_numeric, "choice": score_choice}


def _resolve(scorer: Any) -> Callable[..., bool]:
    if callable(scorer):
        return scorer
    if scorer in _SCORERS:
        return _SCORERS[scorer]
    raise ValueError(f"알 수 없는 채점 방식: {scorer!r}. 가능: {list(_SCORERS)} 또는 함수")


# --------------------------------------------------------------------------
# 실행 / Running the evaluation
# --------------------------------------------------------------------------
def run_eval(
    items: list[GoldenItem],
    run_fn: Callable[[str], Any],
    *,
    repeats: int = 5,
    condition: str = "base",
    verbose: bool = True,
    on_error: str = "wrong",
) -> Any:
    """각 문항을 `repeats` 회 실행하고 채점한다.

    EN — Run every item `repeats` times and score each run.

    Parameters
    ----------
    run_fn : 질문을 받아 답변(문자열 또는 객체)을 돌려주는 함수
             EN — a function taking the question and returning the answer
    repeats : 문항당 반복 횟수. **1이면 안 된다.** 변동을 못 본다
              EN — repetitions per item; NEVER 1, or you see no variability
    on_error : 실행이 예외로 죽었을 때 "wrong"(오답 처리) 또는 "skip"
               EN — on an exception, either count it wrong or skip the trial

    Returns
    -------
    DataFrame : item_id, rep, condition, correct, answer, elapsed, error
                **long format** — 한 행이 한 번의 시행
                EN — long format: one row per trial
    """
    import time

    import pandas as pd

    행 = []
    for item in items:
        점수함수 = _resolve(item.scorer)
        for rep in range(1, repeats + 1):
            t0 = time.perf_counter()
            answer, err = None, None
            try:
                answer = run_fn(item.question)
            except Exception as exc:
                err = f"{type(exc).__name__}: {exc}"
                if on_error == "skip":
                    continue
            경과 = time.perf_counter() - t0

            correct = False
            if err is None:
                try:
                    correct = bool(점수함수(answer, item.expected,
                                          tolerance=item.tolerance))
                except Exception as exc:
                    err = f"채점 실패: {type(exc).__name__}: {exc}"

            행.append({
                "item_id": item.id, "rep": rep, "condition": condition,
                "correct": correct, "answer": str(answer)[:500],
                "elapsed": round(경과, 2), "error": err,
                **{f"tag_{k}": v for k, v in item.tags.items()},
            })
            if verbose:
                print("O" if correct else ("E" if err else "X"), end="", flush=True)
        if verbose:
            print(f"  {item.id}")
    return pd.DataFrame(행)


# --------------------------------------------------------------------------
# 집계 / Aggregation
# --------------------------------------------------------------------------
def wilson(successes: int, n: int, alpha: float = 0.05) -> tuple[float, float]:
    """이항비율의 Wilson 신뢰구간.

    n이 작거나 비율이 0/1에 가까울 때 정규근사(Wald)보다 훨씬 낫다.
    Wald 는 [0,1] 을 벗어나거나 폭이 0이 되어 버린다.

    EN — The Wilson interval for a binomial proportion. Far better than the
    normal (Wald) approximation when n is small or the proportion is near 0 or 1,
    where Wald escapes [0,1] or collapses to zero width.
    """
    if n == 0:
        return (0.0, 1.0)
    from scipy.stats import norm

    z = norm.ppf(1 - alpha / 2)
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


def per_item(df: Any) -> Any:
    """문항별 정확도. **어떤 문항이 어려운지 보는 것이 지표 하나보다 중요하다.**

    EN — Per-item accuracy. SEEING WHICH ITEMS ARE HARD MATTERS MORE THAN ANY
    SINGLE HEADLINE NUMBER.
    """
    import pandas as pd

    g = df.groupby("item_id", sort=False)["correct"]
    out = pd.DataFrame({"n": g.count(), "맞음": g.sum(), "정확도": g.mean()})
    out["항상맞음"] = out["정확도"] == 1.0
    out["항상틀림"] = out["정확도"] == 0.0
    return out.reset_index()


def summarize(df: Any, alpha: float = 0.05) -> dict[str, Any]:
    """정확도를 **군집 보정과 함께** 요약한다.

    ★ 이 함수가 이 모듈의 핵심이다. ★

    같은 문항을 반복 실행한 관측은 서로 독립이 아니다.
    그래서 두 가지를 함께 보고한다.

      naive_ci   : 모든 시행을 독립으로 본 Wilson 구간 (대부분의 AI 평가 코드)
      cluster_ci : 문항을 군집으로 본 구간 (문항별 정확도의 표본평균 기반)

    `design_effect` 가 1보다 크면 naive 구간이 **실제보다 좁다**는 뜻이다.
    즉 없는 차이를 있다고 말할 위험이 있다.

    EN — Summarise accuracy WITH THE CLUSTER CORRECTION. This is the heart of
    the module. Repeated runs of the same item are not independent, so we report
    both intervals: `naive_ci` is the Wilson interval treating every trial as
    independent (what most AI evaluation code does), and `cluster_ci` treats
    each item as a cluster (based on the sample mean of per-item accuracies).
    A `design_effect` above 1 means the naive interval IS NARROWER THAN THE
    TRUTH — the setup where you claim a difference that is not there.
    """
    import numpy as np
    from scipy.stats import t as t_dist

    n_trials = len(df)
    n_correct = int(df["correct"].sum())
    p_hat = n_correct / n_trials if n_trials else 0.0

    # (1) 순진한 구간 — 모든 시행이 독립이라고 가정
    #     EN — the naive interval, assuming every trial is independent
    naive = wilson(n_correct, n_trials, alpha)

    # (2) 군집 보정 — 문항별 정확도를 하나의 관측으로 본다
    #     EN — cluster correction: one per-item accuracy is one observation
    문항정확도 = df.groupby("item_id", sort=False)["correct"].mean().values
    m = len(문항정확도)
    p_cluster = float(문항정확도.mean()) if m else 0.0
    if m > 1:
        se = float(문항정확도.std(ddof=1) / math.sqrt(m))
        tcrit = t_dist.ppf(1 - alpha / 2, df=m - 1)
        cluster = (max(0.0, p_cluster - tcrit * se), min(1.0, p_cluster + tcrit * se))
    else:
        se, cluster = float("nan"), (0.0, 1.0)

    # (3) 급내상관 ICC — 같은 문항 안의 관측이 얼마나 닮았는가
    #     일원 임의효과 모형의 표준 추정량을 쓴다.
    #        ICC = (MSB - MSW) / (MSB + (k-1) MSW)
    #     신뢰구간 폭의 비율로 역산하면 t분포 임계값 때문에 1을 넘어 버린다.
    #     EN — the intraclass correlation: how alike are observations inside one
    #     item. We use the standard one-way random-effects estimator. Recovering
    #     it from the ratio of CI widths would exceed 1 because of the t critical
    #     value.
    icc = _icc_oneway(df)

    # (4) 설계효과 — 표준 공식. 실효표본 = 전체시행 / deff
    #     EN — design effect, standard formula; effective n = trials / deff
    k_bar = n_trials / m if m else 0
    deff = 1 + (k_bar - 1) * icc if not math.isnan(icc) else float("nan")

    naive_width = naive[1] - naive[0]
    cluster_width = cluster[1] - cluster[0]

    return {
        "n_items": m,
        "n_trials": n_trials,
        "reps_per_item": round(k_bar, 1),
        "accuracy": round(p_hat, 4),
        "accuracy_by_item": round(p_cluster, 4),
        "naive_ci": (round(naive[0], 4), round(naive[1], 4)),
        "cluster_ci": (round(cluster[0], 4), round(cluster[1], 4)),
        "cluster_se": round(se, 4) if not math.isnan(se) else None,
        "icc": round(icc, 3) if not math.isnan(icc) else None,
        "design_effect": round(deff, 2) if not math.isnan(deff) else None,
        "effective_n": round(n_trials / deff, 1) if deff and deff > 0 else None,
        "ci_width_ratio": round(cluster_width / naive_width, 2) if naive_width > 0 else None,
        "n_errors": int(df["error"].notna().sum()),
    }


def _icc_oneway(df: Any) -> float:
    """일원 임의효과 급내상관.

        ICC = (MSB - MSW) / (MSB + (k-1) MSW)

    문항이 군집, 반복 시행이 군집 내 관측이다.
    음수가 나오면 0으로 자른다 (군집 효과가 없다는 뜻).

    EN — One-way random-effects intraclass correlation. The item is the cluster
    and the repeated trials are the observations within it. A negative estimate
    is clipped to 0, meaning there is no cluster effect.
    """
    그룹 = [g["correct"].astype(float).values for _, g in df.groupby("item_id", sort=False)]
    m = len(그룹)
    if m < 2:
        return float("nan")
    n_i = [len(g) for g in 그룹]
    if min(n_i) < 2:
        return float("nan")

    전체평균 = sum(g.sum() for g in 그룹) / sum(n_i)
    ss_b = sum(len(g) * (g.mean() - 전체평균) ** 2 for g in 그룹)
    ss_w = sum(((g - g.mean()) ** 2).sum() for g in 그룹)
    df_b, df_w = m - 1, sum(n_i) - m
    if df_w <= 0:
        return float("nan")
    ms_b, ms_w = ss_b / df_b, ss_w / df_w

    # 균형이 안 맞을 때의 유효 군집 크기 / effective cluster size when unbalanced
    N = sum(n_i)
    k0 = (N - sum(x * x for x in n_i) / N) / (m - 1)
    if k0 <= 0 or (ms_b + (k0 - 1) * ms_w) == 0:
        return float("nan")
    icc = (ms_b - ms_w) / (ms_b + (k0 - 1) * ms_w)
    return float(min(1.0, max(0.0, icc)))


def consistency(df: Any) -> dict[str, Any]:
    """같은 문항을 반복했을 때 얼마나 일관되는가.

    정확도가 같아도 성질이 전혀 다를 수 있다.

      A: 문항의 절반은 늘 맞고 절반은 늘 틀림  → 정확도 50%, 일관성 100%
      B: 모든 문항을 반반씩 맞힘              → 정확도 50%, 일관성 50%

    A는 "못 푸는 문제가 정해져 있다" 이고 B는 "매번 운이다" 이다.
    **고쳐야 할 것이 완전히 다르다.**

    EN — How consistent is the agent across repeats of the same item? Two runs
    can share an accuracy and be nothing alike: in A half the items are always
    right and half always wrong (50% accurate, 100% consistent); in B every item
    is right half the time (50% accurate, 50% consistent). A means "there is a
    fixed set of problems it cannot solve"; B means "it is luck every time".
    WHAT YOU HAVE TO FIX IS COMPLETELY DIFFERENT.
    """
    import numpy as np

    문항 = df.groupby("item_id", sort=False)["correct"]
    정확도 = 문항.mean()
    # 다수결과 일치하는 비율 (0.5 -> 0.5, 0 또는 1 -> 1.0)
    # EN — agreement with the majority verdict
    다수결일치 = 정확도.apply(lambda p: max(p, 1 - p))
    return {
        "항상 맞음": int((정확도 == 1.0).sum()),
        "항상 틀림": int((정확도 == 0.0).sum()),
        "들쭉날쭉": int(((정확도 > 0) & (정확도 < 1)).sum()),
        "평균 다수결 일치율": round(float(다수결일치.mean()), 3),
        "문항 간 정확도 표준편차": round(float(정확도.std(ddof=1)), 3) if len(정확도) > 1 else None,
    }


# --------------------------------------------------------------------------
# 조건 비교 / Comparing conditions
# --------------------------------------------------------------------------
def compare_conditions(
    df_a: Any, df_b: Any, *, name_a: str = "A", name_b: str = "B", alpha: float = 0.05
) -> dict[str, Any]:
    """두 조건(프롬프트 A/B 등)을 비교한다.

    **같은 문항으로 두 조건을 돌렸으므로 짝지은 자료다.**
    독립 두 표본 검정을 쓰면 문항 난이도 차이가 오차에 섞여 검정력을 잃는다.
    4주차에 "대응표본을 독립표본으로 다루지 마라" 고 한 것과 같은 이야기다.

    두 가지 방법을 함께 준다.
      - 문항별 다수결을 이분값으로 만들어 **McNemar 검정**
      - 문항별 정확도 차이에 **대응표본 t / Wilcoxon**

    EN — Compare two conditions (prompt A vs prompt B, say). BOTH CONDITIONS RAN
    ON THE SAME ITEMS, SO THE DATA IS PAIRED. An independent two-sample test
    would fold item difficulty into the error term and throw away power — the
    same point as "do not treat paired data as independent" in week 4. Two
    approaches are reported: McNemar's test on the per-item majority verdict,
    and a paired t / Wilcoxon test on the per-item accuracy differences.
    """
    import pandas as pd
    from scipy.stats import ttest_rel, wilcoxon

    a = df_a.groupby("item_id", sort=False)["correct"].mean()
    b = df_b.groupby("item_id", sort=False)["correct"].mean()
    공통 = a.index.intersection(b.index)
    if len(공통) < 2:
        return {"error": "공통 문항이 2개 미만입니다."}
    a, b = a.loc[공통], b.loc[공통]

    # --- McNemar (문항별 다수결을 정답/오답으로) ---
    # EN — McNemar on the per-item majority verdict, as right/wrong
    a_bin, b_bin = (a > 0.5), (b > 0.5)
    n01 = int((~a_bin & b_bin).sum())     # A만 틀리고 B만 맞음 / wrong in A, right in B
    n10 = int((a_bin & ~b_bin).sum())     # A만 맞고 B만 틀림 / right in A, wrong in B
    if n01 + n10 == 0:
        mcnemar_p = 1.0
    else:
        from scipy.stats import binomtest

        mcnemar_p = float(binomtest(n10, n01 + n10, 0.5).pvalue)

    # --- 문항별 정확도 차이 / per-item accuracy differences ---
    diff = (b - a).values
    if len(diff) > 1 and diff.std(ddof=1) > 0:
        t_res = ttest_rel(b, a)
        t_p = float(t_res.pvalue)
        try:
            w_p = float(wilcoxon(b, a).pvalue)
        except Exception:
            w_p = float("nan")
    else:
        t_p = w_p = 1.0

    return {
        "n_items": len(공통),
        f"{name_a} 정확도": round(float(a.mean()), 4),
        f"{name_b} 정확도": round(float(b.mean()), 4),
        "차이(B-A)": round(float(b.mean() - a.mean()), 4),
        "McNemar_불일치": f"A만맞음 {n10} / B만맞음 {n01}",
        "McNemar_p": round(mcnemar_p, 4),
        "대응t_p": round(t_p, 4),
        "Wilcoxon_p": round(w_p, 4) if not math.isnan(w_p) else None,
        "유의(alpha=%.2f)" % alpha: bool(min(mcnemar_p, t_p) < alpha),
        "주의": "짝지은 자료다. 독립표본 검정을 쓰면 검정력을 잃는다.",
    }


# --------------------------------------------------------------------------
# 표본 크기 / Sample size
# --------------------------------------------------------------------------
def required_n(
    p_expected: float = 0.75,
    half_width: float = 0.10,
    alpha: float = 0.05,
    *,
    icc: float = 0.0,
    repeats: int = 1,
) -> dict[str, Any]:
    """원하는 정밀도를 얻으려면 문항이 몇 개 필요한가.

    `icc` 를 주면 **군집 설계를 반영**한다.
    같은 문항을 여러 번 돌려도 실효 표본은 시행 수만큼 늘지 않는다.

        설계효과 = 1 + (repeats - 1) x icc
        실효표본 = 전체시행 / 설계효과

    이것이 이번 주의 실용적 결론이다.
    **문항 10개 x 20회보다 문항 40개 x 5회가 낫다.**

    EN — How many items do you need for the precision you want? Supplying `icc`
    ACCOUNTS FOR THE CLUSTERED DESIGN: repeating the same item does not grow the
    effective sample by the number of trials. That is this week's practical
    conclusion — 40 items × 5 repeats beats 10 items × 20 repeats.
    """
    from scipy.stats import norm

    z = norm.ppf(1 - alpha / 2)
    n_simple = math.ceil(z**2 * p_expected * (1 - p_expected) / half_width**2)
    deff = 1 + (repeats - 1) * icc
    n_trials = math.ceil(n_simple * deff)
    n_items = math.ceil(n_trials / repeats)

    return {
        "가정": f"정확도 약 {p_expected:.0%}, 반폭 ±{half_width:.0%}, alpha={alpha}",
        "독립가정 시 필요 시행": n_simple,
        "설계효과": round(deff, 2),
        "군집 보정 후 필요 시행": n_trials,
        "반복": repeats,
        "필요 문항 수": n_items,
    }


In [ ]:
%%writefile orchestra.py
"""전산실험2 12주차 : 다중 에이전트와 워크플로.

지금까지 만든 것은 **에이전트 하나**다. 이제 역할을 나눈다.

    Planner  질문을 단계로 쪼갠다
    Analyst  각 단계를 실제로 수행한다 (도구 / 코드)
    Critic   결과를 검토하고 반려한다
    Orchestrator  이들을 엮고, 반려되면 다시 시킨다

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 정직하게 먼저 말할 것

 **다중 에이전트가 항상 나은 것이 아니다.**

   - 느리다      : 역할이 셋이면 호출이 3배 이상
   - 비싸다      : 토큰도 그만큼
   - 오류가 쌓인다 : Planner 가 틀리면 Analyst 가 아무리 잘해도 소용없다
                    각 단계 정확도가 0.8 이어도 3단계면 0.8³ = 0.51

 그런데도 쓰는 이유는 **Critic** 때문이다.
 검토자가 붙으면 "조용히 틀린 답"이 줄어든다.
 이 수업이 계속 강조한 것과 같은 방향이다.

 11주차 하네스로 **직접 측정해서** 판단하라. 믿지 말고 재라.

 EN — An honest word first: MULTI-AGENT IS NOT ALWAYS BETTER. It is slower
 (three roles means at least three times the calls), it costs more tokens in
 proportion, and errors compound — if the Planner is wrong, no amount of skill
 in the Analyst saves the run; at 0.8 accuracy per step, three steps give
 0.8³ = 0.51. The reason to use it anyway is THE CRITIC: with a reviewer
 attached, quietly wrong answers become rarer — the same direction this course
 has pushed all semester. Use the week-11 harness to MEASURE it yourself.
 Don't take it on faith; take the measurement.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

EN — Week 12: multi-agent systems and workflows. Everything so far has been ONE
agent; now we split the roles. Planner breaks the question into steps, Analyst
carries each step out (with tools or code), Critic reviews the result and can
reject it, and Orchestrator wires them together and re-runs on rejection.

사용 예 · Example:
    from orchestra import Orchestrator
    orch = Orchestrator(llm, toolbox, profiler=prof)
    result = orch.run("비료에 따라 수확량이 다른가?")
    result.show()
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any

__all__ = [
    "Message",
    "Plan",
    "Review",
    "OrchestraResult",
    "Planner",
    "Analyst",
    "Critic",
    "Orchestrator",
]


# --------------------------------------------------------------------------
# 에이전트 사이에 오가는 것 / What passes between the agents
# --------------------------------------------------------------------------
@dataclass
class Message:
    """누가 누구에게 무엇을 말했는가. 전체 흐름을 추적하는 단위.

    EN — Who said what to whom: the unit for tracing the whole flow.
    """

    sender: str
    kind: str          # plan / result / review / final / error
    content: Any
    round: int = 0

    def line(self, width: int = 100) -> str:
        본문 = self.content if isinstance(self.content, str) else json.dumps(
            self.content, ensure_ascii=False)
        return f"[R{self.round}] {self.sender:8s} {self.kind:7s} {본문[:width]}"


@dataclass
class Plan:
    """Planner 가 세운 단계 목록. / The list of steps the Planner produced."""

    steps: list[dict[str, Any]] = field(default_factory=list)
    reason: str = ""
    ok: bool = True
    error: str | None = None

    def __len__(self) -> int:
        return len(self.steps)

    def brief(self) -> str:
        return "\n".join(
            f"{i}. {s.get('goal', '?')}" for i, s in enumerate(self.steps, 1)
        )


@dataclass
class Review:
    """Critic 의 판정. / The Critic's verdict."""

    approved: bool
    problems: list[str] = field(default_factory=list)
    suggestion: str = ""
    raw: str = ""

    def brief(self) -> str:
        if self.approved:
            return "승인"
        return "반려: " + "; ".join(self.problems[:3])


@dataclass
class OrchestraResult:
    """전체 실행 기록. / The record of the whole run."""

    question: str
    plan: Plan | None = None
    answer: str | None = None
    reviews: list[Review] = field(default_factory=list)
    messages: list[Message] = field(default_factory=list)
    rounds: int = 0
    approved: bool = False
    n_llm_calls: int = 0
    elapsed: float = 0.0

    def show(self, width: int = 110) -> None:
        print(f"질문: {self.question}")
        print("=" * 78)
        for m in self.messages:
            print(m.line(width))
        print("=" * 78)
        print(f"라운드 {self.rounds} | 승인 {self.approved} | "
              f"LLM 호출 {self.n_llm_calls}회 | {self.elapsed:.1f}초")
        print(f"최종 답변: {str(self.answer)[:400]}")

    def to_dict(self) -> dict[str, Any]:
        return {
            "question": self.question,
            "answer": self.answer,
            "approved": self.approved,
            "rounds": self.rounds,
            "n_llm_calls": self.n_llm_calls,
            "elapsed": round(self.elapsed, 2),
            "plan": self.plan.steps if self.plan else [],
            "reviews": [{"approved": r.approved, "problems": r.problems}
                        for r in self.reviews],
        }


# --------------------------------------------------------------------------
# 역할 1 — Planner / Role 1 — the Planner
# --------------------------------------------------------------------------
PLANNER_SYSTEM = """\
너는 통계 분석 계획을 세우는 역할이다. 직접 분석하지 않는다.

사용자의 질문을 **실행 가능한 단계**로 쪼갠다.
단계는 1~4개로 하고, 각 단계는 도구 하나로 처리할 수 있을 만큼 작아야 한다.

규칙:
- 데이터에 실제로 있는 열 이름만 쓴다.
- 검정을 고르기 전에 데이터를 확인하는 단계를 먼저 넣는다.
- 질문이 단순하면 단계를 억지로 늘리지 않는다. 1단계면 1단계다.

출력은 JSON 하나뿐이다:
{"steps": [{"goal": "무엇을 할지 한 문장"}, ...], "reason": "왜 이렇게 나눴는지"}"""


class Planner:
    """질문을 단계로 쪼갠다. / Breaks the question into steps."""

    name = "Planner"

    def __init__(self, llm: Any, profiler: Any = None, *, max_steps: int = 4) -> None:
        self.llm = llm
        self.profiler = profiler
        self.max_steps = max_steps
        self.calls = 0

    def plan(self, question: str, feedback: str = "", **kw: Any) -> Plan:
        from structured import JSONParseError, parse_json

        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 600)

        데이터 = f"## 데이터\n{self.profiler.to_prompt()}\n\n" if self.profiler else ""
        되먹임 = f"\n## 이전 계획에 대한 지적\n{feedback}\n계획을 고쳐서 다시 세워라.\n" if feedback else ""

        try:
            self.calls += 1
            원문 = self.llm.chat(
                [{"role": "system", "content": PLANNER_SYSTEM},
                 {"role": "user", "content": f"{데이터}## 질문\n{question}\n{되먹임}\nJSON만 출력하라."}],
                **kw)
            data = parse_json(원문)
            steps = data.get("steps") or []
            if not isinstance(steps, list) or not steps:
                return Plan(ok=False, error="steps 가 비었습니다.")
            steps = [s if isinstance(s, dict) else {"goal": str(s)}
                     for s in steps][: self.max_steps]
            return Plan(steps=steps, reason=str(data.get("reason", "")))
        except JSONParseError as exc:
            return Plan(ok=False, error=f"계획 파싱 실패: {exc}")
        except Exception as exc:
            return Plan(ok=False, error=f"{type(exc).__name__}: {exc}")


# --------------------------------------------------------------------------
# 역할 2 — Analyst / Role 2 — the Analyst
# --------------------------------------------------------------------------
class Analyst:
    """단계를 실제로 수행한다.

    안에서 6주차 `ReActAgent` 를 그대로 쓴다.
    **이미 만든 것을 감싸는 것**이지 새로 만드는 것이 아니다.

    EN — Actually carries out a step. Internally it just uses the week-6
    `ReActAgent`: this WRAPS what we already built rather than building anew.
    """

    name = "Analyst"

    def __init__(self, llm: Any, toolbox: Any, *, max_steps: int = 6) -> None:
        from react import ReActAgent

        self.agent = ReActAgent(llm, toolbox, max_steps=max_steps, verbose=False)
        self.llm = llm
        self.calls = 0

    def execute(self, goal: str, context: str = "", **kw: Any) -> dict[str, Any]:
        질문 = f"{context}\n\n{goal}" if context else goal
        trace = self.agent.run(질문, **kw)
        self.calls += len(trace.steps) + 1
        return {
            "goal": goal,
            "answer": trace.final_answer,
            "success": trace.success,
            "tools": trace.tools_used,
            "n_failed": trace.n_failed,
            "observations": [s.observation for s in trace.steps if s.ok][-3:],
        }


# --------------------------------------------------------------------------
# 역할 3 — Critic / Role 3 — the Critic
# --------------------------------------------------------------------------
CRITIC_SYSTEM = """\
너는 통계 분석 결과를 검토하는 역할이다. 분석을 다시 하지 않는다. 검토만 한다.

다음을 확인한다:
1. 질문에 실제로 답했는가
2. 근거가 되는 수치(p값, 표본 수, 통계량)가 제시되었는가
3. 검정 선택이 데이터의 성질과 맞는가
4. 유의하지 않은 결과를 "차이가 없다"고 단정하지 않았는가
5. 상관을 인과로 서술하지 않았는가
6. 숫자를 지어낸 흔적이 없는가 (도구 결과에 없는 수치)

엄격하게 보되, **트집을 잡지는 마라.**
답이 맞고 근거가 있으면 승인한다.

출력은 JSON 하나뿐이다:
{"approved": true/false, "problems": ["문제1", ...], "suggestion": "무엇을 고쳐야 하는지"}"""


class Critic:
    """분석 결과를 검토하고 반려한다.

    **다중 에이전트를 쓰는 진짜 이유가 이것이다.**
    검토자가 붙으면 "조용히 틀린 답"이 줄어든다.

    EN — Reviews the analysis and can send it back. THIS IS THE REAL REASON TO
    GO MULTI-AGENT: with a reviewer attached, quietly wrong answers get rarer.
    """

    name = "Critic"

    def __init__(self, llm: Any, *, strict: bool = True, system: str = CRITIC_SYSTEM) -> None:
        self.llm = llm
        self.strict = strict
        self.system = system
        self.calls = 0

    def review(self, question: str, answer: str, evidence: str = "", **kw: Any) -> Review:
        from structured import JSONParseError, parse_json

        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 500)

        prompt = (f"## 원래 질문\n{question}\n\n"
                  f"## 분석가의 답변\n{answer}\n\n"
                  f"## 도구 실행 결과 (근거)\n{evidence[:2500]}\n\n"
                  f"검토하고 JSON만 출력하라.")
        try:
            self.calls += 1
            원문 = self.llm.chat(
                [{"role": "system", "content": self.system},
                 {"role": "user", "content": prompt}], **kw)
            data = parse_json(원문)
            problems = data.get("problems") or []
            if not isinstance(problems, list):
                problems = [str(problems)]
            return Review(
                approved=bool(data.get("approved", False)),
                problems=[str(p) for p in problems],
                suggestion=str(data.get("suggestion", "")),
                raw=원문,
            )
        except JSONParseError as exc:
            # 검토자가 형식을 어기면 **승인으로 처리하지 않는다.**
            # 검증을 못 했는데 통과시키는 것이 가장 나쁘다.
            # EN — if the reviewer breaks the format, DO NOT treat it as
            #      approval: passing something you failed to verify is the
            #      worst outcome of all.
            return Review(approved=False,
                          problems=[f"검토자 응답을 읽을 수 없음: {exc}"],
                          raw=str(exc))
        except Exception as exc:
            return Review(approved=False,
                          problems=[f"검토 실패: {type(exc).__name__}: {exc}"])


# --------------------------------------------------------------------------
# 오케스트레이터 / The orchestrator
# --------------------------------------------------------------------------
class Orchestrator:
    """Planner → Analyst → Critic 을 엮는다. 반려되면 다시 시킨다.

    EN — Wires Planner → Analyst → Critic together and re-runs on rejection.

    Parameters
    ----------
    max_rounds : Critic 이 반려했을 때 다시 시도할 횟수 상한
                 EN — cap on retries after the Critic rejects
    use_critic : False 면 Critic 을 끄고 단일 에이전트에 가깝게 동작
                 EN — False turns the Critic off, behaving like a single agent
    replan_on_reject : True 면 반려 시 계획부터 다시. False 면 분석만 다시
                 EN — True replans from scratch on rejection; False re-runs
                 only the analysis
    """

    def __init__(
        self,
        llm: Any,
        toolbox: Any,
        *,
        profiler: Any = None,
        max_rounds: int = 2,
        use_critic: bool = True,
        replan_on_reject: bool = False,
        verbose: bool = True,
    ) -> None:
        self.planner = Planner(llm, profiler)
        self.analyst = Analyst(llm, toolbox)
        self.critic = Critic(llm)
        self.llm = llm
        self.max_rounds = max_rounds
        self.use_critic = use_critic
        self.replan_on_reject = replan_on_reject
        self.verbose = verbose

    # ----------------------------------------------------------------
    def run(self, question: str, **kw: Any) -> OrchestraResult:
        """질문을 받아 끝까지 돌린다. **예외를 밖으로 던지지 않는다.**

        EN — Take a question and run to completion. IT NEVER RAISES.
        """
        import time

        t0 = time.perf_counter()
        self.planner.calls = self.analyst.calls = self.critic.calls = 0
        res = OrchestraResult(question=question)
        feedback = ""

        for rnd in range(1, self.max_rounds + 1):
            res.rounds = rnd

            # --- 1) 계획 / plan ------------------------------------
            if res.plan is None or self.replan_on_reject:
                plan = self.planner.plan(question, feedback, **kw)
                res.plan = plan
                if not plan.ok:
                    res.messages.append(Message("Planner", "error", plan.error, rnd))
                    if self.verbose:
                        print(f"[R{rnd}] Planner 실패: {plan.error}")
                    break
                res.messages.append(Message("Planner", "plan", plan.brief(), rnd))
                if self.verbose:
                    print(f"[R{rnd}] Planner: {len(plan)}단계")
                    for i, s in enumerate(plan.steps, 1):
                        print(f"        {i}. {s.get('goal', '?')[:70]}")

            # --- 2) 실행 / execute ---------------------------------
            결과들, 근거들 = [], []
            맥락 = feedback and f"이전 시도에서 다음 지적을 받았다: {feedback}\n" or ""
            for i, step in enumerate(res.plan.steps, 1):
                out = self.analyst.execute(step.get("goal", ""), 맥락, **kw)
                결과들.append(out)
                근거들.extend(out["observations"])
                res.messages.append(
                    Message("Analyst", "result",
                            f"{i}. {out['goal'][:40]} -> {str(out['answer'])[:60]}", rnd))
                if self.verbose:
                    print(f"        Analyst {i}: {'OK' if out['success'] else '실패'} "
                          f"{str(out['answer'])[:60]}")

            답변 = self._synthesize(question, 결과들, **kw)
            res.answer = 답변
            res.messages.append(Message("Analyst", "final", 답변, rnd))

            # --- 3) 검토 / review ----------------------------------
            if not self.use_critic:
                res.approved = True
                break

            review = self.critic.review(question, 답변, "\n\n".join(근거들), **kw)
            res.reviews.append(review)
            res.messages.append(Message("Critic", "review", review.brief(), rnd))
            if self.verbose:
                print(f"        Critic: {review.brief()[:90]}")

            if review.approved:
                res.approved = True
                break

            feedback = (review.suggestion or "; ".join(review.problems))[:600]
            if rnd == self.max_rounds and self.verbose:
                print(f"        -> {self.max_rounds}라운드 모두 반려됨")

        # 끝까지 반려되면 그 사실을 답변에 남긴다
        # EN — if it was rejected to the end, say so in the answer itself
        if self.use_critic and not res.approved and res.answer:
            지적 = "; ".join(res.reviews[-1].problems[:3]) if res.reviews else ""
            res.answer = (f"{res.answer}\n\n[검토 미통과] 검토자가 다음을 지적했습니다: {지적}\n"
                          f"이 결과를 그대로 사용하지 마십시오.")

        res.n_llm_calls = self.planner.calls + self.analyst.calls + self.critic.calls
        res.elapsed = time.perf_counter() - t0
        return res

    # ----------------------------------------------------------------
    def _synthesize(self, question: str, 결과들: list[dict], **kw: Any) -> str:
        """단계별 결과를 하나의 답변으로 합친다.

        EN — Merge the per-step results into a single answer.
        """
        if len(결과들) == 1:
            return str(결과들[0]["answer"] or "")

        모음 = "\n\n".join(
            f"[단계 {i}] {r['goal']}\n결과: {r['answer']}"
            for i, r in enumerate(결과들, 1))
        try:
            return self.llm.ask(
                f"질문: {question}\n\n아래는 단계별 분석 결과다. "
                f"이것만 근거로 질문에 답하라. 숫자를 지어내지 마라.\n\n{모음}",
                system="너는 분석 결과를 종합해 답하는 역할이다.",
                temperature=0.0, max_tokens=kw.get("max_tokens", 600))
        except Exception:
            return 모음


In [ ]:
!pip install -q openai

# 이 수업 저장소에서 공용 모듈 내려받기
# EN — Load the shared module written by the setup cell above.

from kmu_llm import LLM, ask, check_server

PROVIDER = "kmu"          # 학과 서버가 안 되면 "gemini" 로 바꾸세요 / switch to "gemini" if the department server is down
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

In [ ]:
%%writefile exam_greenhouse.csv
pot_id,fertilizer,greenhouse,yield_before,yield_after,water_liters,temp_c,leaf_count,pest_damage
G0001,C,south,63.63,63.91,2.33,25.8,30.0,No
G0002,C,north,44.55,38.06,2.23,22.1,22.0,no
G0003,A,north,28.47,28.83,2.85,20.5,30.0,no
G0004,B,south,60.01,58.01,0.78,-999,34.0,No
G0005,B,south,68.94,85.16,1.57,25.4,34.0,No
G0006,A,south,41.56,48.18,"2,58",25.1,32.0,no
G0007,A,south,48.84,48.73,2.5,25.0,47.0,No
G0008,A,south,56.79,56.91,1.73,21.4,31.0,No
G0009,A,south,39.05,42.84,2.04,22.7,33.0,No
G0010,A,south,58.79,60.51,2.12,-999,35.0,Yes
G0011,B,north,61.89,71.84,2.01,20.3,27.0,No
G0012,A,north,53.67,61.11,2.89,23.7,32.0,Yes
G0013,A,south,52.87,57.02,3.55,24.6,29.0,Y
G0014,C,north,49.44,43.73,1.72,22.0,49.0,no
G0015,C,south,44.32,51.82,2.98,25.0,36.0,no
G0016,C,north,43.19,41.9,2.32,23.0,26.0,no
G0017,A,south,56.04,56.93,1.97,27.0,28.0,no
G0018,A,south,58.25,68.38,2.25,22.2,22.0,no
G0019,B,south,66.43,81.83,2.32,27.6,33.0,No
G0020,B,north,512.0,67.2,2.56,29.7,27.0,No
G0021,A,south,44.29,42.75,3.57,27.7,32.0,No
G0022,A,north,34.38,34.04,2.06,20.3,24.0,no
G0023,A,south,55.24,62.74,2.59,20.5,33.0,No
G0024,C,north,53.45,60.79,3.19,22.4,26.0,No
G0025,A,south,55.04,61.16,2.89,21.4,26.0,no
G0026,A,south,66.51,66.72,2.26,28.2,36.0,no
G0027,C,north,43.58,50.97,2.79,26.9,26.0,No
G0028,B,south,59.41,67.56,2.62,16.8,30.0,no
G0029,B,north,63.05,69.99,2.97,22.7,36.0,No
G0030,B,north,53.61,60.34,3.61,20.5,37.0,no
G0031,A,north,49.31,50.56,1.38,27.2,34.0,No
G0032,B,south,51.58,61.88,2.52,-999,27.0,No
G0033,B,south,52.52,59.53,2.43,23.6,,No
G0034,B,south,64.49,68.34,2.16,24.9,30.0,no
G0035,C,north,44.66,51.11,2.95,19.3,35.0,no
G0036,C,south,54.42,63.11,3.35,26.2,31.0,No
G0037,C,south,46.66,53.42,1.82,-999,30.0,No
G0038,A,north,47.05,47.01,2.56,22.2,24.0,no
G0039,B,south,69.03,73.57kg,2.6,22.5,19.0,no
G0040,A,south,52.02,56.99,2.2,24.5,34.0,no
G0041,B,north,73.02,83.96,2.35,-999,,No
G0042,B,north,57.19,67.53,2.21,22.3,38.0,No
G0043,C,south,48.89,51.07,3.31,28.4,22.0,no
G0044,C,south,45.51,43.91,2.03,25.5,35.0,No
G0045,C,south,73.1,72.83,2.67,22.4,36.0,no
G0046,A,south,49.77,53.55,2.16,21.1,26.0,no
G0047,A,south,50.02,54.03,1.98,20.2,25.0,no
G0048,B,north,52.5,60.99,2.18,25.8,22.0,No
G0049,C,south,47.19,48.26,2.22,27.9,32.0,No
G0050,C,south,52.92,57.25,"2,28",25.5,39.0,yes
G0051,A,south,52.14,50.21,3.8,24.8,27.0,No
G0052,C,north,58.18,53.26,3.48,21.7,24.0,No
G0053,B,north,60.12,72.63,1.81,21.3,38.0,No
G0054,C,south,48.38,45.24,2.59,-999,17.0,No
G0055,A,south,68.79,71.0,2.87,25.7,27.0,no
G0056,C,south,55.3,54.23,1.96,22.3,40.0,No
G0057,B,north,54.27,59.11,2.5,20.2,30.0,no
G0058,A,north,51.87,51.74,2.79,23.9,39.0,no
G0059,B,south,57.39,72.06,2.1,24.4,26.0,no
G0060,A,north,55.27,59.75,2.22,27.7,31.0,No
G0061,C,south,40.18,46.98,1.55,22.6,34.0,Y
G0062,A,north,42.66,49.74,1.95,25.9,34.0,yes
G0063,C,south,43.26,41.39,2.6,24.1,27.0,No
G0064,A,south,48.12,51.89,1.6,24.6,34.0,No
G0065,B,north,52.48,52.5,1.57,23.5,31.0,No
G0066,A,north,42.64,42.22,2.19,21.4,28.0,No
G0067,B,south,59.28,63.65,"1,91",22.8,37.0,no
G0068,A,north,49.49,51.99,2.58,23.4,40.0,No
G0069,B,south,60.73,65.78,2.81,19.2,31.0,No
G0070,C,north,58.42,63.93,2.28,27.5,30.0,No
G0071,A,south,56.53,55.65,2.68,19.5,40.0,No
G0072,A,north,42.11,44.37,2.54,24.3,31.0,No
G0073,A,north,52.74,51.42,1.51,24.4,23.0,yes
G0074,C,north,50.85,54.03,2.31,25.1,42.0,No
G0075,C,south,47.49,54.63,1.91,27.1,31.0,no
G0076,A,north,43.56,47.87,3.95,22.5,35.0,Yes
G0077,C,south,52.88,49.8,3.15,27.3,37.0,Yes
G0078,B,south,67.53,72.61,2.61,23.4,31.0,yes
G0079,B,north,56.41,60.02,2.77,22.3,31.0,no
G0080,B,north,62.92,58.97,3.46,26.3,28.0,No
G0081,B,south,63.73,66.0,2.28,27.9,36.0,No
G0082,C,south,45.18,46.22,1.4,25.0,28.0,No
G0083,B,south,59.25,67.01,2.2,24.6,30.0,No
G0084,C,south,59.92,67.33,1.85,18.8,30.0,Y
G0085,A,north,52.66,53.71,1.62,25.0,,no
G0086,C,south,47.7,42.38,2.9,28.3,26.0,no
G0087,A,south,60.8,66.25,2.82,27.7,29.0,No
G0088,B,north,68.3,74.46,2.34,27.4,24.0,No
G0089,A,north,41.89,47.62,2.21,24.2,30.0,Yes
G0090,C,north,52.73,55.13,2.61,28.4,29.0,no
G0091,A,north,52.56,56.19,2.46,25.1,35.0,Yes
G0092,B,north,57.11,58.33,1.96,21.5,40.0,yes
G0093,A,north,44.24,47.81,2.67,18.9,28.0,no
G0094,A,south,55.45,57.43,2.66,26.2,27.0,no
G0095,B,north,58.35,64.22,2.01,18.6,44.0,yes
G0096,B,south,59.22,72.91,1.69,23.3,32.0,No
G0097,B,south,56.92,59.72,2.48,25.0,31.0,No
G0098,B,south,71.57,79.2,"3,06",25.6,28.0,No
G0099,C,south,65.59,63.09,2.72,26.9,36.0,yes
G0100,C,south,55.92,53.6,1.83,25.1,37.0,No
G0101,B,north,56.99,55.43,0.61,23.4,28.0,No
G0102,C,south,55.58,53.03,1.95,26.1,21.0,no
G0103,B,south,57.01,69.18,"2,53",23.2,35.0,yes
G0104,B,north,53.63,60.25,1.88,28.0,27.0,No
G0105,C,south,49.91,51.07,2.38,20.6,26.0,No
G0106,C,south,42.83,43.04,1.89,26.9,35.0,yes
G0107,A,north,52.26,47.42,2.62,23.5,29.0,no
G0108,C,north,47.88,49.17,1.72,23.9,36.0,No
G0109,A,south,53.71,55.9,2.51,25.0,31.0,Y
G0110,C,north,46.1,48.32,1.77,25.6,26.0,No
G0111,B,south,57.65,64.51,2.09,23.8,35.0,No
G0112,C,north,54.48,53.27,2.74,25.1,28.0,No
G0113,A,north,54.8,53.37,2.84,23.8,25.0,no
G0114,B,south,61.46,64.15,2.2,24.7,23.0,No
G0115,C,north,37.61,42.91,2.76,24.9,28.0,No
G0116,A,north,47.21,52.37,2.33,30.1,34.0,no
G0117,B,north,56.41,61.96,"3,56",26.9,33.0,Yes
G0118,B,south,64.33,60.61,2.11,22.5,22.0,No
G0119,B,north,55.62,65.12,1.32,26.7,33.0,No
G0120,A,south,56.54,56.58,1.44,21.1,,no
G0121,A,south,55.05,54.88,3.2,21.2,22.0,no
G0122,A,north,49.31,56.34,2.88,26.8,27.0,no
G0123,C,south,61.84,66.2,2.34,25.2,41.0,No
G0124,B,north,48.53,56.65kg,2.04,25.2,25.0,No
G0125,A,south,37.91,39.22,2.66,29.2,30.0,no
G0126,B,north,59.34,65.72,2.84,23.7,33.0,Yes
G0127,B,north,68.35,74.41,2.0,23.3,36.0,No
G0128,A,north,57.22,48.59,1.38,23.7,24.0,no
G0129,A,south,51.69,49.64,3.01,21.3,32.0,No
G0130,C,south,60.8,62.09,1.73,31.1,29.0,No
G0131,B,south,57.66,60.29,2.77,26.8,32.0,no
G0132,A,north,46.24,52.36,2.73,22.2,33.0,yes
G0133,B,north,57.4,55.62,2.07,27.8,27.0,no
G0134,C,south,36.04,45.16,2.53,22.4,32.0,No
G0135,B,south,51.75,58.96,2.59,21.2,30.0,No
G0136,C,south,59.84,58.73,3.11,23.7,30.0,no
G0137,A,south,66.09,60.19,0.98,22.7,33.0,No
G0138,B,south,59.66,63.69,3.35,21.6,41.0,No
G0139,B,south,60.55,64.43,2.05,28.1,22.0,yes
G0140,B,north,68.88,62.12,2.17,17.6,30.0,Yes
G0141,B,south,72.84,78.61,2.87,26.1,36.0,no
G0142,C,north,49.3,52.61,2.61,23.7,40.0,No
G0143,C,north,50.82,55.4,2.04,24.9,28.0,no
G0144,A,south,59.18,61.6,2.45,26.9,32.0,No
G0145,A,south,53.25,64.61,1.31,28.1,900.0,yes
G0146,A,south,52.95,53.67,3.14,21.1,32.0,No
G0147,A,south,50.76,50.9,2.59,27.2,26.0,No
G0148,A,south,64.77,65.06,2.83,23.0,34.0,no
G0149,A,north,45.39,50.96kg,2.27,20.3,34.0,yes
G0150,C,south,46.53,48.82,2.4,23.2,,yes
G0151,A,south,54.04,58.03,1.67,26.2,33.0,No
G0152,B,north,52.18,60.16kg,2.4,21.5,39.0,no
G0153,A,north,40.83,38.31,2.83,19.5,41.0,no
G0154,B,south,64.8,72.95,1.64,24.7,40.0,Yes
G0155,B,north,35.3,54.17,2.3,25.9,34.0,No
G0156,C,north,49.58,46.72,2.66,21.5,25.0,No
G0157,C,south,48.37,51.85,2.61,18.9,25.0,Yes
G0158,A,north,45.83,43.47,2.56,23.4,34.0,Yes
G0159,C,south,41.96,47.06,2.46,27.3,25.0,Yes
G0160,B,north,59.19,63.6,2.72,24.4,34.0,No
G0161,B,south,55.91,67.17,1.38,23.0,30.0,No
G0162,A,north,56.61,60.92,2.78,23.6,29.0,no
G0163,B,north,50.27,57.73,"2,48",24.1,29.0,no
G0164,C,south,59.15,61.58,2.17,24.4,42.0,yes
G0165,B,north,50.53,51.35,2.85,26.0,32.0,No
G0166,B,north,56.1,65.7,2.72,22.1,29.0,no
G0167,A,north,51.51,45.5,1.63,21.5,29.0,no
G0168,A,north,39.29,39.65kg,2.26,-999,26.0,yes
G0169,A,south,45.81,49.61,1.42,17.4,,Yes
G0170,A,south,51.33,60.67,2.61,23.6,25.0,No
G0171,B,north,58.03,65.13,1.56,26.8,33.0,no
G0172,A,south,47.5,41.3,3.7,27.3,28.0,Yes
G0173,C,south,62.38,55.42,1.75,27.3,24.0,no
G0174,A,north,49.02,51.32,2.11,21.8,34.0,No
G0175,A,south,52.3,61.08,1.97,25.7,37.0,No
G0176,B,south,57.87,67.41,"2,91",25.1,22.0,Y
G0177,C,south,68.87,70.77,2.03,28.3,,No
G0178,A,south,57.68,64.22,2.49,29.2,27.0,Yes
G0179,A,north,39.51,44.83,2.64,23.0,27.0,No
G0180,A,north,38.79,37.13,2.65,28.7,39.0,No
G0181,A,north,70.14,65.83kg,2.45,20.8,33.0,No
G0182,C,north,57.1,59.82,1.83,22.4,26.0,No
G0183,B,north,60.78,66.46,2.55,25.1,36.0,Yes
G0184,B,south,67.57,74.94,2.88,-999,37.0,No
G0185,B,north,51.63,51.22,2.99,26.1,37.0,Yes
G0186,C,north,51.58,51.87,2.31,22.7,39.0,No
G0187,A,north,43.53,43.3,2.58,28.3,19.0,No
G0188,B,north,59.97,58.43,3.95,22.9,30.0,No
G0189,A,north,54.2,55.23,2.46,26.9,29.0,No
G0190,C,south,44.75,47.68,2.29,22.7,31.0,No
G0191,B,south,50.15,59.47,2.15,21.5,34.0,no
G0192,A,south,51.78,64.11,2.41,24.5,36.0,no
G0193,A,south,52.26,50.29,2.39,27.6,24.0,No
G0194,B,south,50.26,58.03,2.07,25.5,32.0,No
G0195,B,north,48.43,50.39,2.28,-999,34.0,No
G0196,C,south,47.54,53.9,2.97,26.1,41.0,No
G0197,B,south,58.36,71.86,2.7,-999,32.0,No
G0198,B,north,65.57,71.83,3.17,23.4,18.0,no
G0199,A,south,53.28,54.1,2.63,16.6,23.0,yes
G0200,A,south,45.83,51.69,1.74,24.5,30.0,No
G0201,A,south,50.09,50.18,1.9,23.1,41.0,No
G0202,B,north,63.8,70.25,2.58,-999,31.0,no
G0203,C,south,63.12,57.35,2.77,21.9,33.0,yes
G0204,B,south,57.3,61.12,3.19,24.8,30.0,No
G0205,C,north,38.13,42.29,0.9,18.6,34.0,no
G0206,C,south,51.98,53.82,3.19,25.4,27.0,No
G0207,A,south,52.94,57.02,2.41,25.7,29.0,No
G0208,B,north,56.92,65.96,1.89,25.9,41.0,No
G0209,C,south,65.93,68.41,3.13,20.5,24.0,No
G0210,B,south,57.97,64.46,2.71,26.8,26.0,No


In [ ]:
import pandas as pd
from profiler import DataProfiler
from tools import StatToolbox

prof = DataProfiler(pd.read_csv("exam_greenhouse.csv"))
green = prof.clean(verbose=False)
prof = DataProfiler(green)          # 정리된 데이터로 다시 진단 / re-diagnose the cleaned data
box = StatToolbox(green)
print("데이터", green.shape, "| 도구", len(box), "개")

---
## 1. 전체 흐름 · The whole flow

```
Planner → Analyst(단계별) → 종합 → Critic
                                    ↓ 반려
                              되먹여 다시

Planner → Analyst (per step) → synthesise → Critic
                                    ↓ rejected
                              feed back and retry
```

또 같은 구조입니다. 3주차, 6주차, 7주차, 9주차와 똑같습니다.
**달라진 것은 검증자가 코드가 아니라 다른 에이전트**라는 점입니다.

> **EN** — The same structure yet again: weeks 3, 6, 7 and 9 all had it. **What
> changed is that the verifier is another agent rather than code.**

In [ ]:
from orchestra import Orchestrator

orch = Orchestrator(llm, box, profiler=prof, max_rounds=2, verbose=True)
res = orch.run("비료 종류에 따라 처리 후 수확량에 차이가 있나요?")

In [ ]:
res.show()

---
## 2. 반려가 실제로 일어나게 해 보기 · Making a rejection actually happen

Critic 을 더 엄격하게 만들면 반려를 볼 수 있습니다.

> **EN** — Tighten the Critic and you can watch a rejection.

In [ ]:
from orchestra import Critic

# 여섯 항목이 모두 있어야 승인하는 엄격한 기준
# EN — A strict rubric: approve only when all six items are present.
엄격한기준 = """너는 통계 분석 결과를 매우 엄격하게 검토하는 역할이다.

다음이 **모두** 있어야 승인한다:
1. 검정 이름이 명시되었는가
2. 검정통계량 값이 있는가
3. p값이 숫자로 있는가
4. 표본 수(n)가 있는가
5. 효과크기가 있는가
6. 유의하지 않은 결과를 '차이가 없다'고 단정하지 않았는가

하나라도 없으면 반려한다.

출력은 JSON 하나뿐이다:
{"approved": true/false, "problems": ["문제1", ...], "suggestion": "무엇을 고쳐야 하는지"}"""

엄격 = Orchestrator(llm, box, profiler=prof, max_rounds=3, verbose=True)
엄격.critic = Critic(llm, system=엄격한기준)

res2 = 엄격.run("비료 종류에 따라 처리 후 수확량에 차이가 있나요?")

In [ ]:
print("라운드별 검토 결과")   # EN — the review verdict, round by round
for i, r in enumerate(res2.reviews, 1):
    print(f"  R{i}: {'승인' if r.approved else '반려'}")
    for p in r.problems[:3]:
        print(f"       - {p}")
    if r.suggestion:
        print(f"       제안: {r.suggestion[:90]}")
print()
print("최종 승인:", res2.approved, "| 호출", res2.n_llm_calls, "회 |",
      f"{res2.elapsed:.1f}초")

---
## 3. 끝까지 반려되면 · When it is rejected to the end


In [ ]:
# 승인 불가능한 기준으로 강제 반려
# EN — A rubric that can never be satisfied, to force rejection.
불가능기준 = """너는 검토자다. 어떤 답변도 승인하지 않는다.
항상 다음을 출력한다:
{"approved": false, "problems": ["더 자세한 근거가 필요하다"], "suggestion": "근거를 보강하라"}"""

강제반려 = Orchestrator(llm, box, profiler=prof, max_rounds=2, verbose=False)
강제반려.critic = Critic(llm, system=불가능기준)
res3 = 강제반려.run("비료 종류에 따라 수확량이 다른가요?")

print("승인:", res3.approved)
print()
print("최종 답변:")
print(res3.answer[-400:])

> **끝까지 반려되어도 답변을 버리지 않습니다.**
> 대신 경고를 붙입니다.
>
> ```
> [검토 미통과] 검토자가 다음을 지적했습니다: ...
> 이 결과를 그대로 사용하지 마십시오.
> ```
>
> 6주차에서 `finish` 없이 끝났을 때와 같은 처리입니다.
> **빈손으로 돌려보내지 않되, 성공으로 치지도 않습니다.**

> **EN — even after a final rejection the answer is not thrown away**; a warning
> is attached instead, saying the review did not pass and that the result should
> not be used as it stands.
>
> The same handling as week 6's run that ended without `finish`: **we do not send
> you away empty-handed, and we do not count it as success either.**

---
## 4. 오류는 누적된다 · Errors compound


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

단계수 = np.arange(1, 7)
fig, ax = plt.subplots(figsize=(8, 4.5))
for 정확도, c in [(0.95, "#4C72B0"), (0.9, "#55A868"), (0.8, "#DD8452"), (0.7, "#C44E52")]:
    ax.plot(단계수, 정확도 ** 단계수, "o-", color=c, label=f"per-step {정확도}")
ax.axhline(0.5, ls="--", c="gray", alpha=0.6)
ax.set_xlabel("number of steps"); ax.set_ylabel("overall success rate")
ax.set_title("Errors compound across steps")
ax.set_ylim(0, 1.02); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("각 단계 정확도가 0.8 이라면")
for n in range(1, 5):
    print(f"  {n}단계: {0.8**n:.2f}")
print()
print("★ Planner 가 틀리면 Analyst 가 아무리 잘해도 소용없습니다.")
print("   오히려 열심히 실행해서 **그럴듯하게 틀린 답**이 나옵니다.")

In [ ]:
# 그래서 단계를 최소로 유지한다 / Hence: keep the number of steps to a minimum
단순 = ["비료에 따라 수확량이 다른가요?",                                 # simple questions
       "온실 위치에 따라 수확량이 다른가요?",
       "처리 전후로 수확량이 늘었나요?"]
복합 = ["비료에 따라 다른지 보고, 다르다면 어느 비료끼리 다른지도 알려주세요.",   # compound questions
       "수확량에 영향을 주는 요인을 모두 찾아 주세요."]

from orchestra import Planner
p = Planner(llm, prof)
print("질문 유형별 계획 단계 수\n")
for 유형, 질문들 in [("단순", 단순), ("복합", 복합)]:
    단계들 = []
    for q in 질문들:
        plan = p.plan(q)
        단계들.append(len(plan) if plan.ok else 0)
        print(f"  [{유형}] {len(plan) if plan.ok else '실패'}단계  {q[:44]}")
    print(f"  -> 평균 {np.mean(단계들):.1f}단계\n")

---
## 5. 비용을 재 본다 · Measuring the cost


In [ ]:
import pandas as pd
import time

질문 = "비료 종류에 따라 처리 후 수확량에 차이가 있나요?"
from react import ReActAgent

기록 = []

# 단일 에이전트 / a single agent
t0 = time.perf_counter()
t = ReActAgent(llm, box, max_steps=6, verbose=False).run(질문)
기록.append({"구성": "단일 (ReAct)", "성공": t.success,
            "LLM호출": len(t.steps) + 1, "초": round(time.perf_counter()-t0, 1),
            "답변": str(t.final_answer)[:70]})

# 다중 에이전트, Critic 없이 / 있이 · multi-agent, without and with the Critic
for 이름, use_critic in [("다중 (Critic X)", False), ("다중 (Critic O)", True)]:
    o = Orchestrator(llm, box, profiler=prof, use_critic=use_critic,
                     max_rounds=2, verbose=False)
    r = o.run(질문)
    기록.append({"구성": 이름, "성공": r.approved,
                "LLM호출": r.n_llm_calls, "초": round(r.elapsed, 1),
                "답변": str(r.answer)[:70]})

pd.DataFrame(기록)

> **호출 수와 시간을 보십시오.**
> 다중은 몇 배 비쌉니다. 그만한 값을 하는지는 **측정해서 판단**해야 합니다.
> 다음 노트북에서 합니다.

> **EN** — **Look at the call count and the time.** Multi-agent costs several
> times more, and whether it is worth that **has to be measured, not assumed.**
> The next notebook does exactly that.

---
## 연습문제 · Exercises

**문제 1.** `replan_on_reject=True` 로 두면 반려 시 계획부터 다시 세웁니다.
`False` 와 비교해 승인율과 호출 수가 어떻게 다른가?

**문제 2.** `max_rounds` 를 1, 2, 3 으로 바꿔 승인율과 호출 수를 재시오.
비용 대비 효과가 좋은 값은?

**문제 3.** Critic 이 **계획도** 검토하게 만드시오.
(Planner 직후에 계획 검토를 한 번 넣는다) 오류 누적이 줄어드는가?

**문제 4.** 종합 단계(`_synthesize`)의 프롬프트에서
"숫자를 지어내지 마라" 를 빼면 어떻게 되는가? 20회 돌려 확인하시오.

> **EN — 1.** `replan_on_reject=True` replans from scratch on rejection. How do
> the approval rate and the call count compare against `False`?
>
> **EN — 2.** Measure the approval rate and call count at `max_rounds` 1, 2 and 3.
> Which gives the best return on cost?
>
> **EN — 3.** Have the Critic review **the plan too** (insert one plan review right
> after the Planner). Does error compounding fall?
>
> **EN — 4.** Remove "do not invent numbers" from the synthesis prompt
> (`_synthesize`). What happens? Check over 20 runs.

In [ ]:
# 여기에 직접 작성해 보세요 / Write your own code here

<details>
<summary><b>해설 보기 · Show the solutions</b></summary>

**문제 3.** 계획 검토를 넣으면 **오류 누적의 첫 고리**를 끊을 수 있습니다.
9주차의 `validate_plan` 을 그대로 쓸 수도 있습니다 — 그것은 **규칙 기반**이라
LLM 호출이 없고 빠릅니다.

**규칙으로 되는 것은 규칙으로.** 9주차의 교훈입니다.
Critic 은 규칙으로 못 잡는 것(서술의 적절성, 근거의 충분함)에 씁니다.

**문제 4.** 종합 단계는 **환각이 들어오기 쉬운 자리**입니다.
단계별 결과를 요약하면서 없는 숫자를 만들어 냅니다.
4주차 요약 압축에서 "숫자와 고유명사가 사라진다"고 한 것과 짝을 이룹니다.
사라지기도 하고 생기기도 합니다.

---

**EN — 3.** Reviewing the plan breaks **the first link in the error chain.** You
can reuse week 9's `validate_plan` as is — it is **rule-based**, so it makes no
LLM call and is fast.

**Whatever a rule can do, a rule should do** — the week-9 lesson. Save the Critic
for what rules cannot catch: whether the prose is appropriate, whether the
evidence suffices.

**EN — 4.** The synthesis step is **where hallucination gets in most easily**: it
invents numbers while summarising the per-step results. It pairs with week 4's
"numbers and proper nouns disappear" in summary compression — they disappear, and
they also appear.

</details>

---
## 정리 · Summary

- Planner → Analyst → 종합 → Critic. 반려되면 되먹임
- **검증자가 코드가 아니라 다른 에이전트**인 것이 달라진 점
- 끝까지 반려되면 **경고를 붙이되 답변은 남긴다**
- **오류는 누적된다.** 0.8³ = 0.51. 단계를 최소로
- 다중은 몇 배 비싸다. **측정해서 판단하라**

> **EN**
> - Planner → Analyst → synthesise → Critic, with rejections fed back
> - What changed: **the verifier is another agent, not code**
> - After a final rejection, **attach a warning but keep the answer**
> - **Errors compound** — 0.8³ = 0.51 — so keep the steps to a minimum
> - Multi-agent costs several times more — **measure before you decide**

**다음 노트북** → [`03_single_vs_multi.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week12/03_single_vs_multi.ipynb)